In [18]:
import requests
import pandas as pd
import re
import os
from io import BytesIO, StringIO
from datetime import datetime

# ----------------------------------------------------------------------
# Column cleaning and standardization (exact same logic as original)
# ----------------------------------------------------------------------
def clean_and_standardize(df):
    """Standardize column names and enforce a uniform schema."""
    df = df.copy()
    df.columns = df.columns.str.strip()

    column_mappings = {
        'Date': ['Report Date', 'Date', 'Incident Date', 'Date & Time'],
        'Time': ['Time', 'Incident Time'],
        'Day': ['Day'],
        'Location': ['Location', 'Station', 'Station Name', 'Stop', 'Stop Name'],
        'Incident': ['Incident', 'Code', 'Description'],
        'Min Delay': ['Min Delay', 'Delay', 'Delay Minutes', 'Delay_Minutes'],
        'Min Gap': ['Min Gap', 'Gap', 'Gap Minutes', 'Gap_Minutes'],
        'Route': ['Route', 'Route Number', 'Route No', 'Route_ID'],
        'Line': ['Line'],
        'Direction': ['Direction', 'Bound'],
        'Vehicle': ['Vehicle', 'Vehicle Number', 'Vehicle_No']
    }

    reverse_mapping = {}
    for std_name, possible in column_mappings.items():
        for name in possible:
            reverse_mapping[name] = std_name

    rename_dict = {col: reverse_mapping[col] for col in df.columns if col in reverse_mapping}
    df = df.rename(columns=rename_dict)

    # Handle Line column -> Route, Route Name
    if 'Line' in df.columns and 'Route' not in df.columns:
        def extract_route_info(line_val):
            if pd.isna(line_val):
                return pd.Series([None, None])
            line_str = str(line_val).strip()
            match = re.match(r'^(\d+)(?:\s+(.+))?$', line_str)
            if match:
                return pd.Series([match.group(1), match.group(2) if match.group(2) else ''])
            match_digits = re.match(r'^(\d+)$', line_str)
            if match_digits:
                return pd.Series([match_digits.group(1), ''])
            return pd.Series([line_str, ''])
        df[['Route', 'Route Name']] = df['Line'].apply(extract_route_info)

    if 'Route' in df.columns and 'Route Name' not in df.columns:
        df['Route Name'] = ''

    required_columns = [
        'Date', 'Route', 'Route Name', 'Time', 'Day', 'Location',
        'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle',
        'Incident_Original'
    ]
    for col in required_columns:
        if col not in df.columns:
            df[col] = None

    df = df[required_columns]
    return df


# ----------------------------------------------------------------------
# Download data for a single mode
# ----------------------------------------------------------------------
def load_mode_delay_data(mode_name, package_id, extra_csv=None, force_csv_only=False, skip_year_filter=False):
    """Downloads all resources for a given transit mode, cleans and returns a DataFrame."""
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    current_year = datetime.now().year
    print(f"\n🚋 Processing {mode_name} data (package: {package_id})")

    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception(f"CKAN API request failed for {mode_name}")

    resources = data["result"]["resources"]
    print(f"📦 Found {len(resources)} resource(s)")

    year_pattern = re.compile(r"(19|20)\d{2}")
    allowed_formats = {'csv'} if force_csv_only else {'csv', 'xlsx', 'xls'}

    mode_dfs = []

    for res in resources:
        res_name = res.get("name", "Unnamed")
        datastore_active = res.get("datastore_active", False)

        if not skip_year_filter:
            year_match = year_pattern.search(res_name)
            if not year_match:
                print(f"  ⏭️  {res_name}: no year found, skipping")
                continue
            year = int(year_match.group(0))
            if year < 2014:
                print(f"  ⏭️  {res_name}: year {year} < 2014, skipping")
                continue
        else:
            year = None

        # Determine format
        if datastore_active:
            fmt = 'csv'
        else:
            fmt = res.get("format", "").lower()
            if not fmt:
                url = res.get("url", "")
                if url.endswith('.csv'):
                    fmt = 'csv'
                elif url.endswith(('.xlsx', '.xls')):
                    fmt = 'xlsx'
                else:
                    fmt = 'unknown'

        if fmt not in allowed_formats:
            print(f"  ⏭️  {res_name}: format '{fmt}' not in {allowed_formats}, skipping")
            continue

        # Apply year‑based format filter only if not skipped
        if not skip_year_filter and not force_csv_only:
            if year >= current_year-1 and fmt != 'csv':
                print(f"  ⏭️  {res_name}: skipping XLSX for latest year, only CSV kept")
                continue
            elif year < current_year-1 and fmt not in ('xlsx', 'xls'):
                print(f"  ⏭️  {res_name}: skipping CSV for year < {current_year}, only XLSX kept")
                continue

        if year:
            print(f"  📄 Resource: {res_name} (year={year}, format={fmt}, datastore_active={datastore_active})")
        else:
            print(f"  📄 Resource: {res_name} (format={fmt}, datastore_active={datastore_active})")

        try:
            if datastore_active:
                dump_url = f"{base_url}/datastore/dump/{res['id']}"
                dump_resp = requests.get(dump_url)
                dump_resp.raise_for_status()
                df = pd.read_csv(StringIO(dump_resp.text))
                if 'Incident' in df.columns:
                    df['Incident_Original'] = df['Incident']
            else:
                file_url = res["url"]
                file_resp = requests.get(file_url)
                file_resp.raise_for_status()

                if fmt in ('xlsx', 'xls'):
                    excel_data = pd.read_excel(BytesIO(file_resp.content), sheet_name=None)
                    sheet_dfs = []
                    for sheet_name, sheet_df in excel_data.items():
                        if not sheet_df.empty:
                            if 'Incident' in sheet_df.columns:
                                sheet_df['Incident_Original'] = sheet_df['Incident']
                            sheet_df = clean_and_standardize(sheet_df)
                            sheet_dfs.append(sheet_df)
                    if sheet_dfs:
                        df = pd.concat(sheet_dfs, ignore_index=True)
                    else:
                        print(f"    ⚠️ No data in any sheet, skipping")
                        continue
                else:
                    df = pd.read_csv(StringIO(file_resp.text))
                    if 'Incident' in df.columns:
                        df['Incident_Original'] = df['Incident']

            # Standardize columns (keeps Incident_Original and original Incident)
            df = clean_and_standardize(df)

            # Add transit mode column (will be kept in final output)
            df['Transit'] = mode_name

            mode_dfs.append(df)
            print(f"    ✅ Loaded {len(df)} records")

        except Exception as e:
            print(f"    ❌ Error: {e}")
            continue

    if extra_csv and os.path.isfile(extra_csv):
        print(f"\n  📄 Extra local file: {extra_csv}")
        try:
            df_extra = pd.read_csv(extra_csv)
            if 'Incident' in df_extra.columns:
                df_extra['Incident_Original'] = df_extra['Incident']
            df_extra = clean_and_standardize(df_extra)
            df_extra['Transit'] = mode_name
            mode_dfs.append(df_extra)
            print(f"    ✅ Loaded {len(df_extra)} records from extra file")
        except Exception as e:
            print(f"    ❌ Error reading extra file {extra_csv}: {e}")
    elif extra_csv:
        print(f"\n  ⏭️ Extra file {extra_csv} not found, skipping")

    if not mode_dfs:
        print(f"⚠️ No valid data loaded for {mode_name}")
        return pd.DataFrame()

    mode_combined = pd.concat(mode_dfs, ignore_index=True)
    print(f"✅ {mode_name} total records: {len(mode_combined)}")
    return mode_combined


# ----------------------------------------------------------------------
# Main function: download and merge all modes, return filtered DataFrame
# with original standardized columns + Transit, and no nulls in Route/Date/MinDelay (MinDelay > 0)
# ----------------------------------------------------------------------
def get_clean_ttc_delays():
    """
    Download all TTC delay datasets (Bus, Streetcar, Subway, LRT),
    merge them, filter rows where Route, Date, and Min Delay are not null
    and Min Delay > 0, and return only the original standardized columns plus Transit.
    """
    modes = [
        ("Bus", "ttc-bus-delay-data"),
        ("Streetcar", "ttc-streetcar-delay-data"),
        ("Subway", "ttc-subway-delay-data"),
        ("LRT", "ttc-lrt-delay-data")
    ]

    all_dfs = []
    for mode_name, pkg_id in modes:
        # For LRT, download all resources without year filtering (as in original)
        if mode_name == "LRT":
            df_mode = load_mode_delay_data(mode_name, pkg_id,
                                           extra_csv="TTC LRT Delays.csv",
                                           force_csv_only=False,
                                           skip_year_filter=True)
        else:
            df_mode = load_mode_delay_data(mode_name, pkg_id)
        if not df_mode.empty:
            all_dfs.append(df_mode)

    if not all_dfs:
        raise Exception("No data loaded for any mode.")

    merged_df = pd.concat(all_dfs, ignore_index=True)
    print(f"\n📊 Total merged records (before filtering): {len(merged_df)}")

    # Drop duplicates
    initial_len = len(merged_df)
    merged_df = merged_df.drop_duplicates()
    print(f"🧹 Removed {initial_len - len(merged_df)} duplicate rows")

    # Keep only the original standardized columns plus Transit.
    # Original standardized columns: Date, Time, Day, Location, Incident,
    # Min Delay, Min Gap, Route, Direction, Vehicle.
    # We also keep Transit.
    cols_to_keep = ['Date', 'Time', 'Day', 'Location', 'Incident',
                    'Min Delay', 'Min Gap', 'Route', 'Direction', 'Vehicle',
                    'Transit']
    # Ensure only columns that actually exist are selected (some may be missing in some datasets)
    cols_to_keep = [col for col in cols_to_keep if col in merged_df.columns]
    merged_df = merged_df[cols_to_keep]

    # Filter rows: Route, Date, Min Delay not null, and Min Delay > 0
    before_filter = len(merged_df)
    merged_df = merged_df.dropna(subset=['Route', 'Date', 'Min Delay'])
    merged_df = merged_df[merged_df['Min Delay'] > 0]
    print(f"🔍 Kept {len(merged_df)} rows after filtering (removed {before_filter - len(merged_df)} rows with null Route/Date/MinDelay or MinDelay <= 0)")

    return merged_df


# ----------------------------------------------------------------------
# Example usage
# ----------------------------------------------------------------------
if __name__ == "__main__":
    df = get_clean_ttc_delays()
    print("\n👀 First few rows (original columns + Transit):")
    print(df.head())
    print("\n📋 Columns returned:", list(df.columns))


🚋 Processing Bus data (package: ttc-bus-delay-data)
📦 Found 20 resource(s)
  ⏭️  ttc-bus-delay-data-readme: no year found, skipping
  📄 Resource: ttc-bus-delay-data-2014 (year=2014, format=xlsx, datastore_active=False)
    ✅ Loaded 94217 records
  📄 Resource: ttc-bus-delay-data-2015 (year=2015, format=xlsx, datastore_active=False)
    ✅ Loaded 76510 records
  📄 Resource: ttc-bus-delay-data-2016 (year=2016, format=xlsx, datastore_active=False)
    ✅ Loaded 77088 records
  📄 Resource: ttc-bus-delay-data-2017 (year=2017, format=xlsx, datastore_active=False)
    ✅ Loaded 70303 records
  📄 Resource: ttc-bus-delay-data-2018 (year=2018, format=xlsx, datastore_active=False)
    ✅ Loaded 73927 records
  📄 Resource: ttc-bus-delay-data-2019 (year=2019, format=xlsx, datastore_active=False)
    ✅ Loaded 62376 records
  📄 Resource: ttc-bus-delay-data-2020 (year=2020, format=xlsx, datastore_active=False)
    ✅ Loaded 36151 records
  📄 Resource: ttc-bus-delay-data-2021 (year=2021, format=xlsx, datast

In [19]:
df_all = df.copy()

In [20]:
df_all['Date'] = pd.to_datetime(df_all['Date'])
df_all['Date'].max()

Timestamp('2026-05-31 00:00:00')

In [21]:
df.head()

,Date,Time,Day,Location,Incident,Min Delay,Min Gap,Route,Direction,Vehicle,Transit
0,2014-01-01 00:00:00,00:23:00,Wednesday,York Mills station,Mechanical,10.0,20.0,95,E,1734.0,Bus
1,2014-01-01 00:00:00,00:55:00,Wednesday,Entire run for route,General Delay,33.0,66.0,102,b/w,8110.0,Bus
2,2014-01-01 00:00:00,01:28:00,Wednesday,lawrence and Warden,Mechanical,10.0,20.0,54,WB,7478.0,Bus
3,2014-01-01 00:00:00,01:30:00,Wednesday,Kipling Station,Emergency Services,18.0,36.0,112,N,8084.0,Bus
4,2014-01-01 00:00:00,01:37:00,Wednesday,VP and Ellesmere,Investigation,10.0,20.0,24,n,7843.0,Bus


In [22]:
# 1. Convert 'Date' to datetime (coerce errors to NaT)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# 2. Extract useful components from the date
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Weekday'] = df['Date'].dt.day_name()

# 3. Extract hour from the 'Time' column (handles "HH:MM" and "HH:MM:SS")
def extract_hour(time_val):
    if pd.isna(time_val):
        return None
    try:
        # Split by ':' and take the first part (hour)
        hour = int(str(time_val).split(':')[0])
        return hour
    except (ValueError, AttributeError):
        return None

df['Hour'] = df['Time'].apply(extract_hour)

# 4. Convert Location column to all uppercase
df['Location'] = df['Location'].str.upper()

In [23]:
import pandas as pd
import re

# ----------------------------------------------------------------------
# Build the master mapping from incident codes to intermediate categories
# ----------------------------------------------------------------------
def _build_code_mapping():
    mapping = {}

    # ---- Subway codes (EU, MU, PU, SU, TU) ----
    # Equipment / Mechanical (EU)
    eu_mech = [
        'EUAC', 'EUAL', 'EUATC', 'EUBK', 'EUBO', 'EUCA', 'EUCH', 'EUCO',
        'EUDO', 'EUECD', 'EUHV', 'EULT', 'EULV', 'EUNEA', 'EUNT', 'EUO',
        'EUPI', 'EUSC', 'EUTL', 'EUTM', 'EUTR', 'EUTRD', 'EUVA', 'EUVE', 'EUYRD'
    ]
    for code in eu_mech:
        mapping[code] = 'Equipment / Mechanical'

    mapping['EUCD'] = 'General Delay / Other'          # Consequential Delay
    mapping['EUME'] = 'Operations / Human Error'       # Maintenance Error
    mapping['EUOE'] = 'Operations / Human Error'       # RC&S Operator Error
    mapping['EUOPO'] = 'Infrastructure / Track / Signals'

    # Miscellaneous (MU)
    mu_mapping = {
        'MUD': 'Passenger / Security',
        'MUDD': 'External / Environment',
        'MUEC': 'Infrastructure / Track / Signals',
        'MUESA': 'Operations / Human Error',
        'MUFM': 'External / Environment',
        'MUFS': 'External / Environment',
        'MUGD': 'General Delay / Other',
        'MUI': 'Passenger / Security',
        'MUIE': 'Passenger / Security',
        'MUIR': 'Passenger / Security',
        'MUIRS': 'Passenger / Security',
        'MUIS': 'Passenger / Security',
        'MULD': 'Management / Administrative',
        'MUNOA': 'Operations / Human Error',
        'MUO': 'General Delay / Other',
        'MUODC': 'Infrastructure / Track / Signals',
        'MUPAA': 'Passenger / Security',
        'MUPLA': 'External / Environment',
        'MUPLB': 'External / Environment',
        'MUPLC': 'External / Environment',
        'MUPR1': 'Passenger / Security',
        'MUSAN': 'Cleaning / Unsanitary',
        'MUSC': 'Equipment / Mechanical',
        'MUTD': 'Management / Administrative',
        'MUTO': 'General Delay / Other',
        'MUWEA': 'External / Environment',
        'MUWR': 'Management / Administrative'
    }
    mapping.update(mu_mapping)

    # Infrastructure (PU)
    pu_infra = [
        'PUATC', 'PUCBI', 'PUCSC', 'PUCSS', 'PUDCS', 'PUMEL', 'PUMO',
        'PUOPO', 'PUSAC', 'PUSBE', 'PUSCA', 'PUSCR', 'PUSEA', 'PUSI',
        'PUSIO', 'PUSIS', 'PUSLC', 'PUSO', 'PUSRA', 'PUSSW', 'PUSTC',
        'PUSTP', 'PUSTS', 'PUSWZ', 'PUSZC', 'PUTCD', 'PUTD', 'PUTIJ',
        'PUTNT', 'PUTO', 'PUTOE', 'PUTR', 'PUTS', 'PUTSC', 'PUTSM',
        'PUTTC', 'PUTTP', 'PUTWZ'
    ]
    for code in pu_infra:
        mapping[code] = 'Infrastructure / Track / Signals'

    mapping['PUMST'] = 'Passenger / Security'
    mapping['PUTDN'] = 'External / Environment'
    mapping['PUTIS'] = 'External / Environment'
    mapping['PUSNT'] = 'General Delay / Other'

    # Security (SU)
    su_sec = [
        'SUAE', 'SUAP', 'SUBT', 'SUCOL', 'SUDP', 'SUEAS', 'SUG',
        'SUO', 'SUPOL', 'SUROB', 'SUSA', 'SUSP', 'SUUT'
    ]
    for code in su_sec:
        mapping[code] = 'Passenger / Security'

    # Transportation (TU)
    tu_mapping = {
        'TUATC': 'Operations / Human Error',
        'TUCC': 'Operations / Human Error',
        'TUDOE': 'Operations / Human Error',
        'TUKEY': 'Operations / Human Error',
        'TUML': 'Scheduling / Late Starts',
        'TUMVS': 'Operations / Human Error',
        'TUNIP': 'Operations / Human Error',
        'TUNOA': 'Operations / Human Error',
        'TUO': 'General Delay / Other',
        'TUOPO': 'Operations / Human Error',
        'TUOS': 'Operations / Human Error',
        'TUS': 'Scheduling / Late Starts',
        'TUSC': 'Operations / Human Error',
        'TUSET': 'Operations / Human Error',
        'TUST': 'External / Environment',
        'TUSUP': 'Operations / Human Error',
        'TUUR': 'Operations / Human Error'
    }
    mapping.update(tu_mapping)

    # ---- Streetcar codes (ER, MR, PR, SR, TR) ----
    # Equipment / Mechanical (ER)
    er_mech = [
        'ERAC', 'ERBO', 'ERCO', 'ERDB', 'ERDO', 'ERHV', 'ERLT', 'ERLV',
        'ERNEA', 'ERNT', 'ERO', 'ERPR', 'ERRA', 'ERTB', 'ERTC', 'ERTL',
        'ERTR', 'ERVE', 'ERWA', 'ERWS'
    ]
    for code in er_mech:
        mapping[code] = 'Equipment / Mechanical'
    mapping['ERCD'] = 'General Delay / Other'
    mapping['ERME'] = 'Operations / Human Error'

    # Miscellaneous (MR)
    mr_mapping = {
        'MRCL': 'Management / Administrative',
        'MRD': 'Passenger / Security',
        'MRDD': 'External / Environment',
        'MREC': 'Infrastructure / Track / Signals',
        'MRESA': 'Operations / Human Error',
        'MRFS': 'External / Environment',
        'MRIE': 'Passenger / Security',
        'MRLD': 'Management / Administrative',
        'MRNOA': 'Operations / Human Error',
        'MRO': 'General Delay / Other',
        'MRPAA': 'Passenger / Security',
        'MRPLA': 'External / Environment',
        'MRPLB': 'External / Environment',
        'MRPLC': 'External / Environment',
        'MRPR1': 'Passenger / Security',
        'MRSAN': 'Cleaning / Unsanitary',
        'MRSTM': 'Infrastructure / Track / Signals',
        'MRTO': 'General Delay / Other',
        'MRUI': 'Passenger / Security',
        'MRUIR': 'Passenger / Security',
        'MRWEA': 'External / Environment'
    }
    mapping.update(mr_mapping)

    # Infrastructure (PR)
    pr_infra = [
        'PREL', 'PRS', 'PRSA', 'PRSL', 'PRSO', 'PRSP', 'PRSW', 'PRTST', 'PRW'
    ]
    for code in pr_infra:
        mapping[code] = 'Infrastructure / Track / Signals'
    mapping['PRO'] = 'General Delay / Other'
    mapping['PRST'] = 'Passenger / Security'

    # Security (SR)
    sr_sec = ['SRAE', 'SRAP', 'SRBT', 'SRCOL', 'SRDP', 'SREAS', 'SRO', 'SRSA', 'SRSP', 'SRUT']
    for code in sr_sec:
        mapping[code] = 'Passenger / Security'

    # Transportation (TR)
    tr_mapping = {
        'TRDOE': 'Operations / Human Error',
        'TRNIP': 'Operations / Human Error',
        'TRNOA': 'Operations / Human Error',
        'TRO': 'General Delay / Other',
        'TRSET': 'Operations / Human Error',
        'TRST': 'External / Environment',
        'TRTC': 'Operations / Human Error'
    }
    mapping.update(tr_mapping)

    # ---- LRT codes (EX, MX, PX, SX, TX) ----
    # Equipment / Mechanical (EX)
    ex_mech = [
        'EXAC', 'EXADD', 'EXBK', 'EXBO', 'EXCB', 'EXCE', 'EXCO', 'EXDB',
        'EXDO', 'EXECD', 'EXGA', 'EXGF', 'EXHV', 'EXLT', 'EXNEA', 'EXNT',
        'EXO', 'EXOSC', 'EXSA', 'EXSE', 'EXTB', 'EXTM', 'EXTR', 'EXVC',
        'EXVE', 'EXWA', 'EXWM', 'EXWS', 'EXYRD'
    ]
    for code in ex_mech:
        mapping[code] = 'Equipment / Mechanical'
    mapping['EXOE'] = 'Operations / Human Error'
    mapping['EXPD'] = 'Collision / Roadblock'
    mapping['EXPI'] = 'Collision / Roadblock'

    # Miscellaneous (MX)
    mx_mapping = {
        'MXAFR': 'Infrastructure / Track / Signals',
        'MXCL': 'Management / Administrative',
        'MXCSA': 'Operations / Human Error',
        'MXD': 'Passenger / Security',
        'MXDD': 'External / Environment',
        'MXESA': 'Operations / Human Error',
        'MXFM': 'External / Environment',
        'MXFS': 'External / Environment',
        'MXGD': 'General Delay / Other',
        'MXI': 'Passenger / Security',
        'MXIC': 'Passenger / Security',
        'MXIE': 'Passenger / Security',
        'MXIR': 'Passenger / Security',
        'MXIRS': 'Passenger / Security',
        'MXIS': 'Passenger / Security',
        'MXLDC': 'Management / Administrative',
        'MXLDT': 'Management / Administrative',
        'MXNCA': 'Operations / Human Error',
        'MXNOA': 'Operations / Human Error',
        'MXO': 'General Delay / Other',
        'MXPAA': 'Passenger / Security',
        'MXPF': 'Infrastructure / Track / Signals',
        'MXPLA': 'External / Environment',
        'MXPLB': 'External / Environment',
        'MXPLC': 'External / Environment',
        'MXPR': 'External / Environment',
        'MXPR1': 'Passenger / Security',
        'MXPU': 'Operations / Human Error',
        'MXSAN': 'Cleaning / Unsanitary',
        'MXTD': 'Management / Administrative',
        'MXTO': 'General Delay / Other',
        'MXUS': 'Scheduling / Late Starts',
        'MXWEA': 'External / Environment',
        'MXWR': 'Management / Administrative'
    }
    mapping.update(mx_mapping)

    # Infrastructure (PX)
    px_infra = [
        'PXATC', 'PXDCS', 'PXEAS', 'PXMEL', 'PXOV', 'PXSAC', 'PXSBE',
        'PXSCA', 'PXSCR', 'PXSI', 'PXSIS', 'PXSRA', 'PXSTP', 'PXSW',
        'PXTD', 'PXTR', 'PXTS', 'PXW', 'PXWZ'
    ]
    for code in px_infra:
        mapping[code] = 'Infrastructure / Track / Signals'
    mapping['PXEME'] = 'Operations / Human Error'
    mapping['PXEO'] = 'General Delay / Other'
    mapping['PXMO'] = 'General Delay / Other'
    mapping['PXMST'] = 'Passenger / Security'
    mapping['PXSNT'] = 'General Delay / Other'
    mapping['PXSO'] = 'General Delay / Other'
    mapping['PXTDN'] = 'External / Environment'
    mapping['PXTIS'] = 'External / Environment'

    # Security (SX)
    sx_sec = [
        'SXAE', 'SXAM', 'SXAP', 'SXAX', 'SXBT', 'SXCOL', 'SXDP',
        'SXEAS', 'SXG', 'SXO', 'SXPOL', 'SXROB', 'SXSA', 'SXSP', 'SXUEG'
    ]
    for code in sx_sec:
        mapping[code] = 'Passenger / Security'

    # Transportation (TX)
    tx_mapping = {
        'TXATC': 'Operations / Human Error',
        'TXCC': 'Operations / Human Error',
        'TXDOE': 'Operations / Human Error',
        'TXLF': 'Scheduling / Late Starts',
        'TXML': 'Scheduling / Late Starts',
        'TXMVS': 'Operations / Human Error',
        'TXNCA': 'Operations / Human Error',
        'TXNIP': 'Operations / Human Error',
        'TXNOA': 'Operations / Human Error',
        'TXO': 'General Delay / Other',
        'TXOI': 'Passenger / Security',
        'TXOS': 'Operations / Human Error',
        'TXOVS': 'Operations / Human Error',
        'TXPD': 'Collision / Roadblock',
        'TXPI': 'Collision / Roadblock',
        'TXS': 'Scheduling / Late Starts',
        'TXST': 'External / Environment',
        'TXSUP': 'Operations / Human Error',
        'TXSV': 'Operations / Human Error'
    }
    mapping.update(tx_mapping)

    # ---- Bus codes (EF, MF, PF, SF, TF) ----
    # Equipment (EF)
    ef_mech = [
        'EFB', 'EFD', 'EFDB', 'EFHV', 'EFHVA', 'EFLV', 'EFP', 'EFRA', 'EFT', 'EFTB'
    ]
    for code in ef_mech:
        mapping[code] = 'Equipment / Mechanical'
    mapping['EFCAN'] = 'General Delay / Other'
    mapping['EFO'] = 'General Delay / Other'

    # Miscellaneous (MF)
    mf_mapping = {
        'MFCN': 'Cleaning / Unsanitary',
        'MFDV': 'Operations / Human Error',
        'MFESA': 'Operations / Human Error',
        'MFFD': 'Passenger / Security',
        'MFLD': 'Management / Administrative',
        'MFO': 'General Delay / Other',
        'MFPI': 'Collision / Roadblock',
        'MFPR': 'External / Environment',
        'MFS': 'External / Environment',
        'MFSAN': 'Cleaning / Unsanitary',
        'MFSH': 'Operations / Human Error',
        'MFTO': 'Operations / Human Error',
        'MFUI': 'Passenger / Security',
        'MFUIR': 'Passenger / Security',
        'MFUS': 'Scheduling / Late Starts',
        'MFVIS': 'Operations / Human Error',
        'MFWEA': 'External / Environment'
    }
    mapping.update(mf_mapping)

    # Plant (PF)
    mapping['PFO'] = 'General Delay / Other'
    mapping['PFPD'] = 'Collision / Roadblock'

    # Security (SF)
    sf_sec = [
        'SFAE', 'SFAP', 'SFBT', 'SFDP', 'SFO', 'SFPOL', 'SFSA', 'SFSP'
    ]
    for code in sf_sec:
        mapping[code] = 'Passenger / Security'

    # Transportation (TF)
    tf_mapping = {
        'TFCNO': 'Operations / Human Error',
        'TFLF': 'Scheduling / Late Starts',
        'TFLL': 'Scheduling / Late Starts',
        'TFO': 'Operations / Human Error',
        'TFOI': 'Passenger / Security',
        'TFPD': 'Collision / Roadblock',
        'TFPI': 'Collision / Roadblock'
    }
    mapping.update(tf_mapping)

    return mapping

_INCIDENT_CODE_MAP = _build_code_mapping()

# ----------------------------------------------------------------------
# Translate intermediate categories to final clean names
# ----------------------------------------------------------------------
CATEGORY_TRANSLATION = {
    'Equipment / Mechanical': 'Mechanical',
    'Operations / Human Error': 'Operations',
    'Infrastructure / Track / Signals': 'Infrastructure',
    'Passenger / Security': 'Passenger',
    'External / Environment': 'External',
    'Scheduling / Late Starts': 'Scheduling',
    'Collision / Roadblock': 'Collision',
    'Cleaning / Unsanitary': 'Cleaning',
    'Management / Administrative': 'Management',
    'General Delay / Other': 'General'
}

# ----------------------------------------------------------------------
# Direct mapping for common free‑text phrases (optional, can be extended)
# ----------------------------------------------------------------------
DIRECT_TEXT_MAP = {
    'Mechanical': 'Mechanical',
    'General Delay': 'General',
    'Emergency Services': 'Passenger',
    'Investigation': 'Passenger',
    'Diversion': 'Operations',
    'Late Leaving Garage': 'Scheduling',
    'Utilized Off Route': 'Operations',
    'Vision': 'Operations',
    'Late Leaving Garage - Operator': 'Operations',
    'Late Leaving Garage - Mechanical': 'Mechanical',
    'Late Leaving Garage - Management': 'Management',
    'Late Leaving Garage - Vision': 'Operations',
    'Management': 'Management',
    'Operations - Operator': 'Operations',
    'Cleaning': 'Cleaning',
    'Security': 'Passenger',
    'Collision - TTC': 'Collision',
    'Road Blocked - NON-TTC Collision': 'Collision',
    'Road Block - Non-TTC Collision': 'Collision',
    'Roadblock by Collision - Non-TTC': 'Collision',
    'Securitty': 'Passenger',
    'Late Entering Service - Mechanical': 'Mechanical',
    'Held By': 'Passenger',
    'Late Leaving Garage - Operations': 'Operations',
    'e': 'General',
    'Late Entering Service': 'Scheduling',
    'Operations': 'Operations',
    'Cleaning - Unsanitary': 'Cleaning',
    'Cleaning - Disinfection': 'Cleaning',
    'Collision - TTC Involved': 'Collision',
    'Late': 'Scheduling',
    'Overhead': 'Infrastructure',
    'Rail/Switches': 'Infrastructure',
    'Overhead - Pantograph': 'Infrastructure',
    'Late  ': 'Scheduling'
    # Add any other frequently seen phrases here
}

# ----------------------------------------------------------------------
# Main mapping function
# ----------------------------------------------------------------------
def map_to_category(incident):
    """
    Convert any incident value (code or description) into one of ten categories:
    Mechanical, Operations, Infrastructure, Passenger, External,
    Scheduling, Collision, Cleaning, Management, General.
    """
    if pd.isna(incident):
        return 'General'
    s = str(incident).strip()
    if not s:
        return 'General'

    # Step 1: If it looks like a standardized code (all caps, possibly with digits)
    if re.match(r'^[A-Z0-9]{2,8}$', s):
        intermediate = _INCIDENT_CODE_MAP.get(s)
        if intermediate:
            return CATEGORY_TRANSLATION.get(intermediate, 'General')
        # If code not in map, fall through to other checks

    # Step 2: Direct match in the text map
    if s in DIRECT_TEXT_MAP:
        return DIRECT_TEXT_MAP[s]

    # Step 3: Keyword‑based pattern matching
    s_lower = s.lower()
    patterns = [
        (r'late (leaving|entering)|unable to maintain schedule|mainline storage', 'Scheduling'),
        (r'collision|road ?block', 'Collision'),
        (r'clean|unsanitary|disinfection', 'Cleaning'),
        (r'management|clerk|training|labour dispute|work refusal', 'Management'),
        (r'weather|ice|snow|fire|debris|force majeure|storm', 'External'),
        (r'mechanical|equipment|brakes|door.*faulty|hvac|propulsion', 'Mechanical'),
        (r'operations?.*operator|signal violation|overshot|overspeed|not in position|supervisory', 'Operations'),
        (r'passenger|security|assault|disorderly|bomb|alarm|unauthorized|injur', 'Passenger'),
        (r'infrastructure|track|signal|power|escalator|elevator|switch|rail|debris.*controllable', 'Infrastructure'),
    ]
    for pattern, category in patterns:
        if re.search(pattern, s_lower):
            return category

    # Step 4: Default
    return 'General'

df['Incident_Category'] = df['Incident'].apply(map_to_category)

In [24]:
import re

def clean_route(val):
    """
    Convert route values to integer route numbers:
    - Direct replacements: YU→1, BD→2, SRT→3, SHP→985, FW→6
    - Otherwise, extract the first integer found in the string.
    - If no integer found, return None.
    """
    if pd.isna(val):
        return None
    s = str(val).strip()
    
    # Direct replacements for subway line codes
    replacement_map = {
        'YU': 1,
        'BD': 2,
        'SRT': 3,
        'SHP': 4,
        'FW': 6
    }
    if s in replacement_map:
        return replacement_map[s]
    
    # Extract first integer (including those in strings like "32.0")
    match = re.search(r'\d+', s)
    if match:
        return int(match.group())
    
    # No number found – leave as None (or you could keep original string)
    return None

# Apply to the Route column (updates in place)
df['Route'] = df['Route'].apply(clean_route)
df = df[df['Route'].notnull()]

In [25]:
bus = df[df['Transit']=='Bus']
sub = df[df['Transit']=='Subway']
strt = df[df['Transit']=='Streetcar']
lrt = df[df['Transit']=='LRT']

In [26]:
import requests
import pandas as pd
import zipfile
from io import BytesIO, StringIO

# ----------------------------------------------------------------------
# Function to download routes.txt only
# ----------------------------------------------------------------------
def download_routes_gtfs():
    """Download routes.txt from the TTC GTFS package and return as DataFrame."""
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    package_id = "merged-gtfs-ttc-routes-and-schedules"

    # Get package metadata
    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception("CKAN API request failed for GTFS package")

    # Find the ZIP resource (complete GTFS)
    zip_resource = None
    for res in data["result"]["resources"]:
        name = res.get("name", "").lower()
        fmt = res.get("format", "").lower()
        if "gtfs" in name and fmt == "zip":
            zip_resource = res
            break

    if not zip_resource:
        raise Exception("No GTFS ZIP resource found in package")

    print(f"📥 Downloading GTFS ZIP from: {zip_resource['url']}")
    zip_resp = requests.get(zip_resource['url'])
    zip_resp.raise_for_status()

    # Extract routes.txt
    with zipfile.ZipFile(BytesIO(zip_resp.content)) as zf:
        if 'routes.txt' in zf.namelist():
            with zf.open('routes.txt') as f:
                routes_df = pd.read_csv(f)
            print("✅ Extracted routes.txt")
        else:
            raise Exception("routes.txt not found in ZIP")

    return routes_df

# ----------------------------------------------------------------------
# Download routes.txt
# ----------------------------------------------------------------------
routes_df = download_routes_gtfs()

# Display a preview
print("\n📋 routes.txt columns:", list(routes_df.columns))
print(routes_df[['route_id', 'route_short_name', 'route_long_name']].head())

# ----------------------------------------------------------------------
# Prepare for merging
# ----------------------------------------------------------------------
# Ensure both key columns are clean
# Convert Route to integer where possible (to match route_short_name)
# First, drop rows where Route is missing
df = df.dropna(subset=['Route'])

# Convert Route to integer (if it's a float like 95.0, it becomes 95)
df['Route'] = pd.to_numeric(df['Route'], errors='coerce').astype('Int64')

# Convert route_short_name to integer as well (GTFS stores them as strings but they are numeric)
routes_df['route_short_name'] = pd.to_numeric(routes_df['route_short_name'], errors='coerce').astype('Int64')

# Drop rows where route_short_name is missing (just in case)
routes_df = routes_df.dropna(subset=['route_short_name'])

# ----------------------------------------------------------------------
# Debug: Check sample values
# ----------------------------------------------------------------------
print("\nSample values from df['Route'] (first 20):")
print(df['Route'].dropna().head(20).tolist())
print("\nSample values from routes_df['route_short_name'] (first 20):")
print(routes_df['route_short_name'].head(20).tolist())

# ----------------------------------------------------------------------
# Remove any existing 'route_long_name' column to avoid merge conflicts
# ----------------------------------------------------------------------
if 'route_long_name' in df.columns:
    print("⚠️ Removing existing 'route_long_name' column before merge.")
    df = df.drop(columns=['route_long_name'])

# ----------------------------------------------------------------------
# Left join to add route_long_name
# ----------------------------------------------------------------------
# Keep only necessary columns from routes_df to avoid duplication
df = df.merge(routes_df[['route_short_name', 'route_long_name']],
              left_on='Route',
              right_on='route_short_name',
              how='left')

# Drop the temporary key column
df = df.drop(columns=['route_short_name'])

# ----------------------------------------------------------------------
# Check results
# ----------------------------------------------------------------------
print("\n✅ Route names added. Columns now:", list(df.columns))
print("\nSample matches:")
print(df[['Route', 'route_long_name']].drop_duplicates().head(10))

# Count how many rows got a match vs. missing
matched = df['route_long_name'].notna().sum()
total = len(df)
print(f"\n📊 Matched {matched} out of {total} rows ({matched/total*100:.1f}%)")

📥 Downloading GTFS ZIP from: https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/b811ead4-6eaf-4adb-8408-d389fb5a069c/resource/c920e221-7a1c-488b-8c5b-6d8cd4e85eaf/download/completegtfs.zip
✅ Extracted routes.txt

📋 routes.txt columns: ['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_desc', 'route_type', 'route_url', 'route_color', 'route_text_color']
   route_id  route_short_name       route_long_name
0        10                10             Van Horne
1       100               100       Flemingdon Park
2       101               101        Downsview Park
3       102               102            Markham Rd
4       103               103  Mount Pleasant North

Sample values from df['Route'] (first 20):
[95, 102, 54, 112, 24, 129, 36, 53, 36, 320, 91, 129, 96, 54, 12, 12, 54, 320, 35, 32]

Sample values from routes_df['route_short_name'] (first 20):
[10, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 11, 110, 111, 112, 113, 114, 115, 116, 117]

✅ Route names 

In [27]:
df[df['route_long_name'].isnull()]

,Date,Time,Day,Location,Incident,Min Delay,Min Gap,Route,Direction,Vehicle,Transit,Year,Month,Weekday,Hour,Incident_Category,route_long_name
22,2014-01-01,05:40:00,Wednesday,STEELES AND MCCOWAN,Late Leaving Garage,30.0,60.0,302,SB,7788.0,Bus,2014,1,Wednesday,5.0,Scheduling,NaN
23,2014-01-01,05:40:00,Wednesday,60 CLEARVIEW HIGHTS AT TRETHEWEY,Utilized Off Route,1.0,1.0,809,BW,1593.0,Bus,2014,1,Wednesday,5.0,Operations,NaN
24,2014-01-01,05:40:00,Wednesday,60 CLEARVIEW HIGHTS AT TRETHEWEY,Utilized Off Route,1.0,1.0,809,BW,1591.0,Bus,2014,1,Wednesday,5.0,Operations,NaN
31,2014-01-01,06:54:00,Wednesday,T1 - AIRPORT,Late Leaving Garage,20.0,42.0,58,E,1568.0,Bus,2014,1,Wednesday,6.0,Scheduling,NaN
41,2014-01-01,08:00:00,Wednesday,LAWRENCE WEST STATION,Late Leaving Garage,20.0,40.0,58,W,1634.0,Bus,2014,1,Wednesday,8.0,Scheduling,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
975267,2023-07-22,06:11,Saturday,SCARBOROUGH CTR STATIO,MRTO,7,14,3,S,3018,Subway,2023,7,Saturday,6.0,General,NaN
975278,2023-07-23,08:15,Sunday,ELLESMERE STATION,MRTO,5,12,3,N,3001,Subway,2023,7,Sunday,8.0,General,NaN
975279,2023-07-23,13:42,Sunday,MCCOWAN STATION,MRTO,10,17,3,S,3012,Subway,2023,7,Sunday,13.0,General,NaN
975280,2023-07-23,01:08,Sunday,MCCOWAN STATION,ERTC,6,12,3,S,3008,Subway,2023,7,Sunday,1.0,Mechanical,NaN


In [28]:
# ----------------------------------------------------------------------
# Ensure Route is integer (clean up any floats or strings)
# ----------------------------------------------------------------------
# Convert to numeric, coerce errors to NaN, then to nullable integer
df['Route'] = pd.to_numeric(df['Route'], errors='coerce').astype('Int64')

# ----------------------------------------------------------------------
# Manual route name mapping (for routes missing in GTFS or needing override)
# ----------------------------------------------------------------------
route_name_manual = {
    3: "Line 3 (Scarborough RT)",
    4: "Line 4 (Sheppard)",
    56: "Leaside",
    195: "Jane Rocket",
    196: "York University Express",
    199: "Finch Rocket",
    # Add others if needed
}

# ----------------------------------------------------------------------
# Fill missing route_long_name using manual map
# ----------------------------------------------------------------------
mask = df['route_long_name'].isna() & df['Route'].isin(route_name_manual.keys())
df.loc[mask, 'route_long_name'] = df.loc[mask, 'Route'].map(route_name_manual)

# ----------------------------------------------------------------------
# Check results
# ----------------------------------------------------------------------
print("Updated route_long_name for the specified routes:")
print(df[df['Route'].isin([3, 56, 195, 196, 199])][['Route', 'route_long_name']].drop_duplicates())

Updated route_long_name for the specified routes:
       Route          route_long_name
49       199             Finch Rocket
189      196  York University Express
663       56                  Leaside
8518       3  Line 3 (Scarborough RT)
25473    195              Jane Rocket


In [29]:
route_counts = df['Route'].value_counts()

# Identify routes with at least 10 records
routes_to_keep = route_counts[route_counts >= 10].index

# Filter the DataFrame
initial_rows = len(df)
df = df[df['Route'].isin(routes_to_keep)]

# Report
removed_rows = initial_rows - len(df)
print(f"Removed {removed_rows} rows from routes with < 10 records.")
print(f"Remaining rows: {len(df)}")

Removed 391 rows from routes with < 10 records.
Remaining rows: 1002166


In [30]:
subway_routes = [1, 2, 3, 4]
lrt_routes = [5, 6]
streetcar_routes = [301, 304, 305, 306, 310, 312,
                    501, 503, 504, 505, 506, 507, 508, 509, 510, 511, 512]

# Create boolean masks for rows that should be removed
remove_subway = df['Route'].isin(subway_routes) & (df['Transit'] != 'Subway')
remove_lrt = df['Route'].isin(lrt_routes) & (df['Transit'] != 'LRT')
remove_streetcar = df['Route'].isin(streetcar_routes) & (df['Transit'] != 'Streetcar')

# Combine masks – remove if any condition is true
to_remove = remove_subway | remove_lrt | remove_streetcar

# Keep only rows that are not to be removed
df = df[~to_remove].copy()
df.loc[df['Route'] == 985, 'Transit'] = 'Bus'
allowed_streetcar_routes = {
    301, 304, 305, 306, 310, 312,
    501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 512,
    514, 521, 522, 500
}

# Identify rows to remove: Transit is Streetcar but Route not in allowed set
to_remove = (df['Transit'] == 'Streetcar') & (~df['Route'].isin(allowed_streetcar_routes))

# Optional: print how many will be removed
print(f"Removing {to_remove.sum()} rows with Transit='Streetcar' and invalid Route.")

# Keep only the rows that are NOT in to_remove
df = df[~to_remove].copy()

# Define allowed routes
subway_routes = {1, 2, 3, 4}
streetcar_routes = {
    301, 304, 305, 306, 310, 312,
    501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 512,
    514, 521, 522, 500
}

# Masks for invalid combinations
invalid_subway = (df['Transit'] == 'Subway') & (~df['Route'].isin(subway_routes))
invalid_streetcar = (df['Transit'] == 'Streetcar') & (~df['Route'].isin(streetcar_routes))

# Combine masks – remove if either is true
to_remove = invalid_subway | invalid_streetcar

print(f"Removing {to_remove.sum()} rows with inconsistent Transit/Route combinations.")
print(f"  - Invalid Subway: {invalid_subway.sum()} rows")
print(f"  - Invalid Streetcar: {invalid_streetcar.sum()} rows")

# Keep valid rows
df = df[~to_remove].copy()

Removing 975 rows with Transit='Streetcar' and invalid Route.
Removing 7 rows with inconsistent Transit/Route combinations.
  - Invalid Subway: 7 rows
  - Invalid Streetcar: 0 rows


In [31]:
import requests
import pandas as pd
import zipfile
from io import BytesIO, StringIO

# ----------------------------------------------------------------------
# 1. Download trips.txt
# ----------------------------------------------------------------------
def download_trips_gtfs():
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    package_id = "merged-gtfs-ttc-routes-and-schedules"
    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception("CKAN API request failed for GTFS package")

    zip_resource = None
    for res in data["result"]["resources"]:
        name = res.get("name", "").lower()
        fmt = res.get("format", "").lower()
        if "gtfs" in name and fmt == "zip":
            zip_resource = res
            break

    if not zip_resource:
        raise Exception("No GTFS ZIP resource found")

    print(f"📥 Downloading GTFS ZIP from: {zip_resource['url']}")
    zip_resp = requests.get(zip_resource['url'])
    zip_resp.raise_for_status()

    with zipfile.ZipFile(BytesIO(zip_resp.content)) as zf:
        if 'trips.txt' in zf.namelist():
            with zf.open('trips.txt') as f:
                trips_df = pd.read_csv(f)
            print("✅ Extracted trips.txt")
        else:
            raise Exception("trips.txt not found in ZIP")
    return trips_df

# ----------------------------------------------------------------------
# 2. Load trips and build variant mapping
# ----------------------------------------------------------------------
trips_df = download_trips_gtfs()
print(f"Trips loaded: {len(trips_df)} rows")

# Clean GTFS columns
trips_df['route_id'] = trips_df['route_id'].astype(str).str.strip()
trips_df['trip_short_name'] = trips_df['trip_short_name'].fillna('').astype(str).str.strip()

# Keep only rows with non‑empty trip_short_name (these define variants)
variants_df = trips_df[trips_df['trip_short_name'] != '']

# Group by route_id and collect unique trip_short_names
variant_map = variants_df.groupby('route_id')['trip_short_name'].unique().reset_index()
variant_map['trip_short_name'] = variant_map['trip_short_name'].apply(sorted)

# Build dictionary: route_id -> list of full variant strings (e.g., "129A")
route_variants = {}
for _, row in variant_map.iterrows():
    route = row['route_id']
    short_names = row['trip_short_name']
    route_variants[route] = [f"{route}{sn}" for sn in short_names]

print(f"Variants built for {len(route_variants)} routes")

# ----------------------------------------------------------------------
# 3. Prepare df for variant expansion
# ----------------------------------------------------------------------
# First, map known special route codes to numeric strings
special_map = {
    'YU': '1',
    'BD': '2',
    'SRT': '3',
    'SHP': '985',
    'FW': '6'
}
df['Route'] = df['Route'].replace(special_map)

# Convert Route to clean string:
# - If it's a float like 129.0, convert to int then to string '129'
# - If it's already a string, just strip.
def clean_route(val):
    if pd.isna(val):
        return None
    # Convert to string, then remove trailing .0 if present
    s = str(val).strip()
    # If it ends with '.0', remove it
    if s.endswith('.0'):
        s = s[:-2]
    return s
df['OG'] = df['Route']
df['Route'] = df['Route'].apply(clean_route)

# ----------------------------------------------------------------------
# 4. Add variant list and expand
# ----------------------------------------------------------------------
# Create temporary column with list of variants (NaN if none)
df['_variants'] = df['Route'].map(route_variants)

# For rows where _variants is NaN, replace with a list containing the original Route
mask = df['_variants'].isna()
df.loc[mask, '_variants'] = df.loc[mask, 'Route'].apply(lambda x: [x])

# Now explode
df_expanded = df.explode('_variants').reset_index(drop=True)
df_expanded['Route'] = df_expanded['_variants']
df = df_expanded.drop(columns=['_variants'])

print(f"Expanded DataFrame shape: {df.shape}")
print("Sample new Route values:", df['Route'].head(10).tolist())

📥 Downloading GTFS ZIP from: https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/b811ead4-6eaf-4adb-8408-d389fb5a069c/resource/c920e221-7a1c-488b-8c5b-6d8cd4e85eaf/download/completegtfs.zip


C:\Users\bains\AppData\Local\Temp\ipykernel_75572\2955802574.py:37: DtypeWarning: Columns (0: trip_id, 1: trip_short_name) have mixed types. Specify dtype option on import or set low_memory=False.
  trips_df = pd.read_csv(f)


✅ Extracted trips.txt
Trips loaded: 134626 rows
Variants built for 123 routes
Expanded DataFrame shape: (1870234, 18)
Sample new Route values: ['95A', '95B', '102A', '102B', '102C', '102D', '102S', '54A', '54B', '112B']


In [32]:
print("Unique Route values (sample):", df['Route'].dropna().unique()[:20])
print("Route data type:", df['Route'].dtype)
print("Number of rows with non-numeric Route:", df['Route'].astype(str).str.match(r'^\d+$').sum())
print("Total rows:", len(df))

Unique Route values (sample): <StringArray>
[ '95A',  '95B', '102A', '102B', '102C', '102D', '102S',  '54A',  '54B',
 '112B', '112C', '112S',  '24A',  '24B', '129A', '129B', '129S',   '36',
  '53A',  '53B']
Length: 20, dtype: str
Route data type: str
Number of rows with non-numeric Route: 396279
Total rows: 1870234


In [33]:
import os
import pandas as pd

# ----------------------------------------------------------------------
# Keep Route as string (do NOT convert to numeric)
# ----------------------------------------------------------------------
# This code assumes Route is already a string and contains valid identifiers.
# Ensure Year is present (convert if needed)
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')

# ----------------------------------------------------------------------
# Aggregate by Route (as string), Year, Transit, Incident_Category
# ----------------------------------------------------------------------
route_agg = df.groupby(
    ['Route', 'Year', 'Transit', 'Incident_Category'],
    as_index=False,
    dropna=False
).agg(
    Delay_Count=('Min Delay', 'count'),
    Total_Delay_Min=('Min Delay', 'sum')
)

# ----------------------------------------------------------------------
# Compute active_in_2025 (route had >20 incidents in 2025)
# ----------------------------------------------------------------------
incidents_2025 = df[df['Year'] == 2025].groupby('Route').size().reset_index(name='total_2025')
active_routes = incidents_2025[incidents_2025['total_2025'] > 20]['Route'].tolist()
route_agg['active_in_2025'] = route_agg['Route'].isin(active_routes)

# ----------------------------------------------------------------------
# Add route_long_name (if you have this column)
# ----------------------------------------------------------------------
if 'route_long_name' in df.columns:
    route_names = df.dropna(subset=['route_long_name']).groupby('Route')['route_long_name'].first().reset_index()
    route_agg = route_agg.merge(route_names, on='Route', how='left')
    route_agg['route_long_name'] = route_agg['route_long_name'].fillna('Unknown')
else:
    route_agg['route_long_name'] = 'Unknown'

# ----------------------------------------------------------------------
# Filter out any rows where Route is exactly "0" (if that exists)
# ----------------------------------------------------------------------
route_agg = route_agg[route_agg['Route'] != '0']

# ==================== NEW: ADD RANK COLUMN ====================
# Extract base route (leading digits) to group variants
route_agg['Route_base'] = route_agg['Route'].str.extract(r'^(\d+)')[0]

# Sort to ensure consistent ordering (variant order within each base)
route_agg = route_agg.sort_values(
    ['Route_base', 'Year', 'Transit', 'Incident_Category', 'route_long_name', 'Route']
)

# Assign rank within each group of (base, year, transit, category, long_name)
route_agg['rank'] = route_agg.groupby(
    ['Route_base', 'Year', 'Transit', 'Incident_Category', 'route_long_name']
).cumcount() + 1

# Drop the temporary base column
route_agg = route_agg.drop(columns=['Route_base'])
# ================================================================

# ----------------------------------------------------------------------
# Save to assets/data/route_analysis.csv
# ----------------------------------------------------------------------
output_dir = os.path.join('assets', 'data')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'route_analysis.csv')
route_agg.to_csv(output_path, index=False)

print(f"✅ route_analysis.csv saved to {output_path}")
print(f"Shape: {route_agg.shape}")
print(route_agg.head())

✅ route_analysis.csv saved to assets\data\route_analysis.csv
Shape: (24250, 9)
  Route  Year Transit Incident_Category  Delay_Count Total_Delay_Min  \
0     1  2014  Subway          Cleaning           55             245   
1     1  2014  Subway          External          244            2436   
2     1  2014  Subway           General          297            1330   
3     1  2014  Subway    Infrastructure          275            2752   
4     1  2014  Subway        Management           27             141   

   active_in_2025        route_long_name  rank  
0            True  Yonge-University Line     1  
1            True  Yonge-University Line     1  
2            True  Yonge-University Line     1  
3            True  Yonge-University Line     1  
4            True  Yonge-University Line     1  


In [34]:
df2 = df.copy()

In [35]:
df2['Route'] = df2['OG']
df2 = df2[['Date', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap',
       'Route', 'Direction', 'Vehicle', 'Transit', 'Year', 'Month', 'Weekday',
       'Hour', 'Incident_Category', 'route_long_name']]
df2= df2.drop_duplicates()

In [36]:
import pandas as pd
import numpy as np
import json
import os
import re
from typing import Dict, List, Tuple, Any, Optional

def generate_dashboard_files(df2, gtfs_folder=None, output_folder='assets/data'):
    """
    Generate five TTC dashboard files from an existing delay DataFrame.

    The files are:
      - weekly_patterns.json        (yearly weekday aggregates with flags)
      - yearly_trends.json           (yearly summary with flags)
      - hourly_frequency_delay.json  (yearly hourly breakdown with flags)
      - top_incident_causes.json     (yearly top incident causes with flags)
      - route_performance.json        (per‑route metrics for 2025, with reliability score)

    Parameters
    ----------
    df2 : pandas.DataFrame
        DataFrame containing TTC bus delay data with columns like:
        Date, Time, Day, Location, Incident, Min Delay, Min Gap, Route,
        Direction, Vehicle, etc.
    gtfs_folder : str, optional
        Path to folder containing routes.txt and trips.txt. If provided,
        route names will be enriched.
    output_folder : str
        Directory where the dashboard files will be saved (default 'assets/data').
    """
    # ----------------------------------------------------------------------
    # 1. Helper functions (replicated from the original)
    # ----------------------------------------------------------------------
    def ensure_folder_exists(folder_path):
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)

    def clean_and_standardize(df):
        """Standardize column names and enforce uniform schema."""
        df = df.copy()
        df.columns = df.columns.str.strip()

        column_mappings = {
            'Date': ['Report Date', 'Date', 'Incident Date', 'Date & Time'],
            'Time': ['Time', 'Incident Time'],
            'Day': ['Day'],
            'Location': ['Location', 'Station', 'Station Name', 'Stop', 'Stop Name'],
            'Incident': ['Incident', 'Code', 'Description'],
            'Min Delay': ['Min Delay', 'Delay', 'Delay Minutes', 'Delay_Minutes'],
            'Min Gap': ['Min Gap', 'Gap', 'Gap Minutes', 'Gap_Minutes'],
            'Route': ['Route', 'Route Number', 'Route No', 'Route_ID'],
            'Line': ['Line'],
            'Direction': ['Direction', 'Bound'],
            'Vehicle': ['Vehicle', 'Vehicle Number', 'Vehicle_No']
        }

        reverse_mapping = {}
        for std_name, poss_names in column_mappings.items():
            for name in poss_names:
                reverse_mapping[name] = std_name

        rename_dict = {col: reverse_mapping[col] for col in df.columns if col in reverse_mapping}
        df = df.rename(columns=rename_dict)

        # Handle Line column
        if 'Line' in df.columns and 'Route' not in df.columns:
            def extract_route_info(line_val):
                if pd.isna(line_val):
                    return pd.Series([None, None])
                line_str = str(line_val).strip()
                match = re.match(r'^(\d+)(?:\s+(.+))?$', line_str)
                if match:
                    return pd.Series([match.group(1), match.group(2) or ''])
                match = re.match(r'^(\d+)$', line_str)
                if match:
                    return pd.Series([match.group(1), ''])
                return pd.Series([line_str, ''])
            df[['Route', 'Route Name']] = df['Line'].apply(extract_route_info)

        if 'Route' in df.columns and 'Route Name' not in df.columns:
            df['Route Name'] = ''

        required = ['Date', 'Route', 'Route Name', 'Time', 'Day', 'Location',
                    'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle']
        for col in required:
            if col not in df.columns:
                df[col] = np.nan

        return df[required]

    def clean_delay_data(delay_df):
        """Clean and convert delay data types."""
        df = delay_df.copy()
        df.columns = [str(col).strip() for col in df.columns]

        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

        if 'Time' in df.columns:
            def extract_hour(t):
                if pd.isna(t):
                    return None
                m = re.search(r'(\d{1,2}):', str(t))
                return int(m.group(1)) if m else None
            df['Hour'] = df['Time'].apply(extract_hour)

        for col in ['Min Delay', 'Min Gap', 'Vehicle']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        if 'Route' in df.columns:
            df['Route'] = df['Route'].astype(str).str.extract(r'^(\d+)')[0]

        if 'Date' in df.columns:
            df['Month'] = df['Date'].dt.month_name().str[:3]
            df['Year'] = df['Date'].dt.year
            df['Weekday'] = df['Date'].dt.day_name()

        # Clean string columns
        for col in ['Time', 'Day', 'Location', 'Incident', 'Direction', 'Route Name']:
            if col in df.columns:
                df[col] = df[col].astype(str).replace({'nan': '', 'None': '', 'NaT': ''}).str.strip()

        # Standardize route name casing
        if 'Route Name' in df.columns:
            def std_name(name):
                if pd.isna(name) or name == '':
                    return name
                name_str = str(name).strip()
                if '-' in name_str:
                    return '-'.join(p.strip().title() for p in name_str.split('-'))
                return name_str.title()
            df['Route Name'] = df['Route Name'].apply(std_name)

        return df

    # ----------------------------------------------------------------------
    # 2. Load GTFS data if folder provided
    # ----------------------------------------------------------------------
    route_name_mapping = {}

    if gtfs_folder and os.path.isdir(gtfs_folder):
        routes_file = os.path.join(gtfs_folder, 'routes.txt')
        if os.path.exists(routes_file):
            routes_df = pd.read_csv(routes_file)
            if 'route_short_name' in routes_df.columns and 'route_long_name' in routes_df.columns:
                for _, row in routes_df.iterrows():
                    route_short = str(row['route_short_name']).strip()
                    route_long = str(row['route_long_name']).strip()
                    route_name_mapping[route_short] = route_long
    else:
        print("GTFS folder not provided or missing – proceeding without GTFS enrichment.")

    # ----------------------------------------------------------------------
    # 3. Clean input data
    # ----------------------------------------------------------------------
    print("Cleaning input delay data...")
    df_clean = clean_delay_data(df2)

    # Apply route name mapping if available
    if route_name_mapping:
        df_clean['Route Name'] = df_clean.apply(
            lambda row: route_name_mapping.get(str(row['Route']), row['Route Name'])
            if pd.isna(row['Route Name']) or row['Route Name'] == ''
            else row['Route Name'],
            axis=1
        )

    # Keep all data (no filtering to 2025 for yearly aggregation)
    df_all = df_clean.copy()
    df_all = df_all[df_all['Min Delay'] > 0]  # only delay incidents

    # Determine latest year present
    years = sorted(df_all['Year'].dropna().unique())
    if len(years) == 0:
        print("No valid years found in data.")
        return
    max_year = int(max(years))

    # ----------------------------------------------------------------------
    # 4. Processing functions for each file
    # ----------------------------------------------------------------------
    def process_yearly_trends(df):
        """Yearly summary with flags."""
        stats = df.groupby('Year')['Min Delay'].agg(['count', 'mean']).reset_index()
        stats.columns = ['year', 'incident_count', 'avg_delay']
        result = []
        for _, row in stats.iterrows():
            entry = {
                'year': int(row['year']),
                'incident_count': int(row['incident_count']),
                'avg_delay': float(row['avg_delay']),
                'last1year': row['year'] == max_year,
                'last2year': row['year'] >= max_year - 1,
                'last5year': row['year'] >= max_year - 4
            }
            result.append(entry)
        return result

    def process_yearly_hourly_frequency(df):
        """For each year, list of hours with incident count and avg delay."""
        result = []
        for year in years:
            df_year = df[df['Year'] == year]
            hourly = []
            for h in range(24):
                hd = df_year[df_year['Hour'] == h]
                cnt = len(hd)
                avg = hd['Min Delay'].mean() or 0
                hourly.append({
                    'hour': h,
                    'hour_label': f"{h:02d}:00",
                    'incident_count': int(cnt),
                    'avg_delay': round(float(avg), 1)
                })
            entry = {
                'year': int(year),
                'last1year': year == max_year,
                'last2year': year >= max_year - 1,
                'last5year': year >= max_year - 4,
                'hourly_data': hourly
            }
            result.append(entry)
        return result

    def process_yearly_top_incident_causes(df, top_n=10):
        """For each year, top incident causes with counts and percentages."""
        result = []
        for year in years:
            df_year = df[df['Year'] == year]
            if 'Incident' not in df_year.columns or df_year.empty:
                entry = {
                    'year': int(year),
                    'last1year': year == max_year,
                    'last2year': year >= max_year - 1,
                    'last5year': year >= max_year - 4,
                    'causes': []
                }
                result.append(entry)
                continue
            cnts = df_year['Incident'].value_counts().head(top_n)
            total = len(df_year)
            causes = []
            for typ, cnt in cnts.items():
                causes.append({
                    'incident_type': str(typ),
                    'count': int(cnt),
                    'percentage': round(cnt / total * 100, 1) if total else 0
                })
            entry = {
                'year': int(year),
                'last1year': year == max_year,
                'last2year': year >= max_year - 1,
                'last5year': year >= max_year - 4,
                'causes': causes
            }
            result.append(entry)
        return result

    def process_yearly_weekly_patterns(df):
        """For each year, aggregate by weekday (incident count and avg delay)."""
        wd_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        result = []
        for year in years:
            df_year = df[df['Year'] == year]
            weekly = []
            for wd in wd_order:
                wd_data = df_year[df_year['Weekday'] == wd]
                cnt = len(wd_data)
                avg = wd_data['Min Delay'].mean() or 0
                weekly.append({
                    'weekday': wd,
                    'incident_count': int(cnt),
                    'avg_delay': round(avg, 1)
                })
            entry = {
                'year': int(year),
                'last1year': year == max_year,
                'last2year': year >= max_year - 1,
                'last5year': year >= max_year - 4,
                'weekly': weekly
            }
            result.append(entry)
        return result

    def process_route_performance(df, min_incidents_2025=20):
        """
        For routes active in 2025 with at least `min_incidents_2025` incidents,
        compute: route, route name, avg delay (2025), most common incident cause (2025),
        total incidents (2025), reliability score, and % change in avg delay 2024→2025.
        """
        df_2025 = df[df['Year'] == 2025]
        if df_2025.empty:
            return []

        # Route stats for 2025
        route_stats_2025 = df_2025.groupby(['Route', 'Route Name']).agg(
            incident_count=('Min Delay', 'count'),
            avg_delay=('Min Delay', 'mean')
        ).reset_index()
        route_stats_2025 = route_stats_2025[route_stats_2025['incident_count'] >= min_incidents_2025]

        # Most common incident cause per route in 2025
        top_cause = df_2025.groupby(['Route', 'Incident']).size().reset_index(name='cnt')
        top_cause = top_cause.loc[top_cause.groupby('Route')['cnt'].idxmax()]
        top_cause = top_cause.set_index('Route')['Incident'].to_dict()

        # 2024 avg delay per route
        df_2024 = df[df['Year'] == 2024]
        avg_2024 = df_2024.groupby('Route')['Min Delay'].mean().to_dict()

        # Compute reliability score (0-100, higher is better)
        # Use percentile ranks of avg_delay and incident_count among 2025 routes
        routes_2025 = route_stats_2025.copy()
        if len(routes_2025) > 1:
            # lower avg_delay is better -> use rank pct where lower value gives higher percentile
            routes_2025['avg_delay_rank'] = routes_2025['avg_delay'].rank(pct=True)
            # lower incident_count is better
            routes_2025['incident_count_rank'] = routes_2025['incident_count'].rank(pct=True)
            # invert so that lower delay/count gives higher score
            routes_2025['reliability'] = 100 * (1 - (routes_2025['avg_delay_rank'] * 0.5 + routes_2025['incident_count_rank'] * 0.5))
        else:
            routes_2025['reliability'] = 50.0  # default if only one route

        result = []
        for _, row in routes_2025.iterrows():
            route = str(row['Route'])
            route_name = row['Route Name']
            if pd.isna(route_name) or route_name == '':
                route_name = f"Route {route}"

            # Most common incident cause
            cause = top_cause.get(route, 'Unknown')

            # 2024 avg delay
            avg_2024_val = avg_2024.get(route, None)
            pct_change = None
            if avg_2024_val and avg_2024_val > 0:
                pct_change = round(((row['avg_delay'] - avg_2024_val) / avg_2024_val) * 100, 1)

            result.append({
                'route': route,
                'route_name': route_name,
                'avg_delay_2025': round(row['avg_delay'], 1),
                'most_common_incident_cause': cause,
                'total_incidents_2025': int(row['incident_count']),
                'reliability_score': round(row['reliability'], 1),
                'avg_delay_change_2024_2025_pct': pct_change
            })

        return result

    # ----------------------------------------------------------------------
    # 5. Generate all datasets
    # ----------------------------------------------------------------------
    print("Generating yearly trends...")
    yearly_trends = process_yearly_trends(df_all)

    print("Generating hourly frequency delay (yearly)...")
    hourly_frequency_delay = process_yearly_hourly_frequency(df_all)

    print("Generating top incident causes (yearly)...")
    top_incident_causes = process_yearly_top_incident_causes(df_all)

    print("Generating weekly patterns (yearly)...")
    weekly_patterns = process_yearly_weekly_patterns(df_all)

    print("Generating route performance for 2025...")
    route_performance = process_route_performance(df_all)

    # ----------------------------------------------------------------------
    # 6. Save the five files
    # ----------------------------------------------------------------------
    ensure_folder_exists(output_folder)
    dashboard_dir = os.path.join(output_folder, 'dashboard')
    ensure_folder_exists(dashboard_dir)

    files = {
        'weekly_patterns.json': weekly_patterns,
        'yearly_trends.json': yearly_trends,
        'hourly_frequency_delay.json': hourly_frequency_delay,
        'top_incident_causes.json': top_incident_causes,
        'route_performance.json': route_performance
    }

    for filename, data in files.items():
        file_path = os.path.join(dashboard_dir, filename)
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, default=str)
        print(f"Saved {filename}")

    print("\n✅ All five requested dashboard files generated successfully!")
    print(f"📁 Output folder: {dashboard_dir}")
df2['Incident'] = df2['Incident_Category']
df2['Route Name'] = df2['route_long_name']   # ensure Route Name column exists
generate_dashboard_files(df2, gtfs_folder='path/to/gtfs', output_folder='assets/data')

GTFS folder not provided or missing – proceeding without GTFS enrichment.
Cleaning input delay data...
Generating yearly trends...
Generating hourly frequency delay (yearly)...
Generating top incident causes (yearly)...
Generating weekly patterns (yearly)...
Generating route performance for 2025...
Saved weekly_patterns.json
Saved yearly_trends.json
Saved hourly_frequency_delay.json
Saved top_incident_causes.json
Saved route_performance.json

✅ All five requested dashboard files generated successfully!
📁 Output folder: assets/data\dashboard


In [37]:
df2

,Date,Time,Day,Location,Incident,Min Delay,Min Gap,Route,Direction,Vehicle,Transit,Year,Month,Weekday,Hour,Incident_Category,route_long_name,Route Name
0,2014-01-01,00:23:00,Wednesday,YORK MILLS STATION,Mechanical,10.0,20.0,95,E,1734.0,Bus,2014,1,Wednesday,0.0,Mechanical,York Mills,York Mills
2,2014-01-01,00:55:00,Wednesday,ENTIRE RUN FOR ROUTE,General,33.0,66.0,102,b/w,8110.0,Bus,2014,1,Wednesday,0.0,General,Markham Rd,Markham Rd
7,2014-01-01,01:28:00,Wednesday,LAWRENCE AND WARDEN,Mechanical,10.0,20.0,54,WB,7478.0,Bus,2014,1,Wednesday,1.0,Mechanical,Lawrence East,Lawrence East
9,2014-01-01,01:30:00,Wednesday,KIPLING STATION,Passenger,18.0,36.0,112,N,8084.0,Bus,2014,1,Wednesday,1.0,Passenger,West Mall,West Mall
12,2014-01-01,01:37:00,Wednesday,VP AND ELLESMERE,Passenger,10.0,20.0,24,n,7843.0,Bus,2014,1,Wednesday,1.0,Passenger,Victoria Park,Victoria Park
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1870229,2026-05-28,15:38,Thursday,JANE FWLRT STOP,Mechanical,6,12,6,E,0,LRT,2026,5,Thursday,15.0,Mechanical,Finch West Line,Finch West Line
1870230,2026-05-28,18:15,Thursday,FINCH WEST FWLRT STATI,General,5,11,6,NaN,0,LRT,2026,5,Thursday,18.0,General,Finch West Line,Finch West Line
1870231,2026-05-29,07:41,Friday,JANE FWLRT STOP,Mechanical,7,14,6,E,0,LRT,2026,5,Friday,7.0,Mechanical,Finch West Line,Finch West Line
1870232,2026-05-29,20:09,Friday,DUNCANWOODS STOP,General,5,15,6,E,6515,LRT,2026,5,Friday,20.0,General,Finch West Line,Finch West Line


In [25]:
df['Route'] = df['OG']
df= df[['Date', 'Time', 'Day', 'Location', 'Incident',
       'Min Delay', 'Min Gap', 'Route', 'Direction', 'Vehicle', 'Transit',
       'Year', 'Month', 'Weekday', 'Hour', 'Incident_Category',
       'route_long_name']]
df = df.drop_duplicates()
df

,Date,Time,Day,Location,Incident,Min Delay,Min Gap,Route,Direction,Vehicle,Transit,Year,Month,Weekday,Hour,Incident_Category,route_long_name
0,2014-01-01,00:23:00,Wednesday,YORK MILLS STATION,Mechanical,10.0,20.0,95,E,1734.0,Bus,2014,1,Wednesday,0.0,Mechanical,York Mills
2,2014-01-01,00:55:00,Wednesday,ENTIRE RUN FOR ROUTE,General Delay,33.0,66.0,102,b/w,8110.0,Bus,2014,1,Wednesday,0.0,General,Markham Rd
7,2014-01-01,01:28:00,Wednesday,LAWRENCE AND WARDEN,Mechanical,10.0,20.0,54,WB,7478.0,Bus,2014,1,Wednesday,1.0,Mechanical,Lawrence East
9,2014-01-01,01:30:00,Wednesday,KIPLING STATION,Emergency Services,18.0,36.0,112,N,8084.0,Bus,2014,1,Wednesday,1.0,Passenger,West Mall
12,2014-01-01,01:37:00,Wednesday,VP AND ELLESMERE,Investigation,10.0,20.0,24,n,7843.0,Bus,2014,1,Wednesday,1.0,Passenger,Victoria Park
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1867622,2026-04-20,09:56,Monday,NORFINCH OAKDALE STOP,PXSO,7,17,6,E,6506,LRT,2026,4,Monday,9.0,General,Finch West Line
1867623,2026-04-20,11:19,Monday,NORFINCH OAKDALE STOP,EXDO,18,28,6,E,6500,LRT,2026,4,Monday,11.0,Mechanical,Finch West Line
1867624,2026-04-20,12:11,Monday,FINCH AND WESTON,MXO,124,130,6,W,6517,LRT,2026,4,Monday,12.0,General,Finch West Line
1867625,2026-04-20,12:32,Monday,EMERY STOP,MXO,20,26,6,E,6502,LRT,2026,4,Monday,12.0,General,Finch West Line


In [38]:
df= df[['Date', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap',
       'Route', 'Direction', 'Vehicle', 'Transit', 'Year', 'Month', 'Weekday',
       'Hour', 'Incident_Category', 'route_long_name']]

In [41]:
import requests
import zipfile
import pandas as pd
import re
import json
import numpy as np
from io import BytesIO
from collections import defaultdict
from rapidfuzz import fuzz, process
from geopy.distance import distance as geopy_distance
from shapely.geometry import LineString, Point, MultiLineString
from shapely.ops import nearest_points
from shapely.errors import GEOSException

# ----------------------------------------------------------------------
# 1. Download GTFS ZIP and extract required files
# ----------------------------------------------------------------------
def download_gtfs_files():
    """
    Download the TTC GTFS ZIP package and extract stops.txt, trips.txt, shapes.txt.
    Returns a dict: {'stops': DataFrame, 'trips': DataFrame, 'shapes': DataFrame}
    """
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    package_id = "merged-gtfs-ttc-routes-and-schedules"

    # Get package metadata
    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception("CKAN API request failed for GTFS package")

    # Find the ZIP resource
    zip_resource = None
    for res in data["result"]["resources"]:
        name = res.get("name", "").lower()
        fmt = res.get("format", "").lower()
        if "gtfs" in name and fmt == "zip":
            zip_resource = res
            break

    if not zip_resource:
        raise Exception("No GTFS ZIP resource found in package")

    print(f"📥 Downloading GTFS ZIP from: {zip_resource['url']}")
    zip_resp = requests.get(zip_resource['url'])
    zip_resp.raise_for_status()

    # Extract required files
    required_files = ['stops.txt', 'trips.txt', 'shapes.txt']
    dataframes = {}
    with zipfile.ZipFile(BytesIO(zip_resp.content)) as zf:
        for file in required_files:
            if file in zf.namelist():
                with zf.open(file) as f:
                    dataframes[file.replace('.txt', '')] = pd.read_csv(f)
                print(f"✅ Extracted {file}")
            else:
                raise Exception(f"{file} not found in ZIP")

    return dataframes

# ----------------------------------------------------------------------
# 2. Text normalization (shared by stops and user locations)
# ----------------------------------------------------------------------
ABBREV_MAP = {
    r'\bav(e)?\b': 'avenue',
    r'\bb(lvd?)\b': 'boulevard',
    r'\brd\b': 'road',
    r'\bdr\b': 'drive',
    r'\bst\b': 'street',
    r'\bhwy\b': 'highway',
    r'\bpkwy\b': 'parkway',
    r'\bcres\b': 'crescent',
    r'\bctr\b': 'centre',
    r'\bctr\b': 'center',
    r'\bpl\b': 'place',
    r'\bcr(t)?\b': 'court',
    r'\bstn\b': 'station',
    r'\blp\b': 'loop',
    r'\bdiv\b': 'division',
    r'\bgar\b': 'garage',
    r'\bmt\b': 'mount',
    r'\bpt\b': 'point',
    r'\bpk\b': 'park',
}

DIR_WORDS = {'north', 'south', 'east', 'west'}

def normalize_text(text, expand_directions=True):
    """Normalize a string: lowercase, expand abbreviations, optionally expand single-letter directions."""
    if pd.isna(text):
        return ''
    s = str(text).lower()
    # Expand abbreviations
    for pattern, repl in ABBREV_MAP.items():
        s = re.sub(pattern, repl, s)
    if expand_directions:
        # Expand single-letter directions (with word boundaries)
        dir_map = {r'\bn\b': 'north', r'\bs\b': 'south', r'\be\b': 'east', r'\bw\b': 'west'}
        for pattern, repl in dir_map.items():
            s = re.sub(pattern, repl, s)
    # Remove punctuation (keep letters, numbers, spaces)
    s = re.sub(r'[^\w\s]', ' ', s)
    # Collapse multiple spaces
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def remove_direction_words(text):
    """Remove direction words (north, south, east, west) from a normalized string."""
    words = text.split()
    words = [w for w in words if w not in DIR_WORDS]
    return ' '.join(words)

# ----------------------------------------------------------------------
# 3. Build stop matcher (unchanged)
# ----------------------------------------------------------------------
def build_stop_matcher(stops_df):
    """
    Prepare data structures for fast matching, including direction-stripped versions.
    Returns a tuple of all needed structures.
    """
    stops = stops_df.copy()
    stops['norm_name'] = stops['stop_name'].apply(lambda x: normalize_text(x, expand_directions=True))
    stops['word_set'] = stops['norm_name'].apply(lambda x: set(x.split()))

    stops['norm_name_no_dir'] = stops['norm_name'].apply(remove_direction_words)
    stops['word_set_no_dir'] = stops['norm_name_no_dir'].apply(lambda x: set(x.split()))

    stop_names = stops['stop_name'].tolist()
    stop_lats = stops['stop_lat'].tolist()
    stop_lons = stops['stop_lon'].tolist()

    # Strict index (with directions)
    stop_norms = stops['norm_name'].tolist()
    stop_words_list = stops['word_set'].tolist()
    word_to_indices = defaultdict(set)
    for idx, words in enumerate(stop_words_list):
        for word in words:
            word_to_indices[word].add(idx)

    # Fallback index (directions removed)
    stop_norms_no_dir = stops['norm_name_no_dir'].tolist()
    stop_words_list_no_dir = stops['word_set_no_dir'].tolist()
    word_to_indices_no_dir = defaultdict(set)
    for idx, words in enumerate(stop_words_list_no_dir):
        for word in words:
            word_to_indices_no_dir[word].add(idx)

    return (stop_names, stop_lats, stop_lons,
            stop_norms, stop_words_list, word_to_indices,
            stop_norms_no_dir, stop_words_list_no_dir, word_to_indices_no_dir)

# ----------------------------------------------------------------------
# 4. Build route geometries from trips and shapes
# ----------------------------------------------------------------------
def build_route_geometries(trips_df, shapes_df):
    """
    Create a dictionary mapping route_id to a list of shapely LineStrings
    (one per shape_id used by that route).
    """
    # Merge trips with shapes to get shape_id per trip
    # We only need distinct (route_id, shape_id) pairs
    route_shapes = trips_df[['route_id', 'shape_id']].drop_duplicates()
    # Convert route_id to string for consistent keys
    route_shapes['route_id'] = route_shapes['route_id'].astype(str)

    # Group shapes by shape_id and create LineString
    shape_lines = {}
    for shape_id, group in shapes_df.groupby('shape_id'):
        # Sort by shape_pt_sequence
        group = group.sort_values('shape_pt_sequence')
        # Convert [lat, lon] to [(lon, lat)] for shapely
        coords = [(lon, lat) for lat, lon in zip(group['shape_pt_lat'], group['shape_pt_lon'])]
        try:
            line = LineString(coords)
            shape_lines[shape_id] = line
        except Exception as e:
            print(f"⚠️ Could not create LineString for shape {shape_id}: {e}")

    # Build route -> list of lines
    route_geoms = defaultdict(list)
    for _, row in route_shapes.iterrows():
        route_id = row['route_id']
        shape_id = row['shape_id']
        if shape_id in shape_lines:
            route_geoms[route_id].append(shape_lines[shape_id])

    # Convert defaultdict to dict
    route_geoms = dict(route_geoms)
    print(f"✅ Built geometries for {len(route_geoms)} routes")
    return route_geoms

def point_to_route_distance(point, route_lines):
    """
    Compute minimum distance in meters from a shapely Point(lon, lat)
    to any valid route shape.

    Defensive version:
    - skips None / empty route geometries
    - skips zero-length lines
    - skips bad coordinates
    - catches Shapely GEOSException from nearest_points()
    """
    if point is None or point.is_empty:
        return float('inf')

    try:
        if not np.isfinite(point.x) or not np.isfinite(point.y):
            return float('inf')
    except Exception:
        return float('inf')

    if route_lines is None:
        return float('inf')

    min_dist = float('inf')

    for line in route_lines:
        if line is None:
            continue

        try:
            if line.is_empty:
                continue
        except Exception:
            continue

        # Handle MultiLineString just in case GTFS/shapely creates one
        if isinstance(line, MultiLineString):
            sub_lines = list(line.geoms)
        else:
            sub_lines = [line]

        for sub_line in sub_lines:
            try:
                if sub_line is None or sub_line.is_empty:
                    continue

                # zero-length / degenerate geometry can break nearest_points
                if sub_line.length == 0:
                    continue

                coords = list(sub_line.coords)
                if len(coords) < 2:
                    continue

                # skip lines with NaN / inf coordinates
                has_bad_coord = False
                for coord in coords:
                    x, y = coord[:2]
                    if not np.isfinite(x) or not np.isfinite(y):
                        has_bad_coord = True
                        break

                if has_bad_coord:
                    continue

                nearest = nearest_points(point, sub_line)[1]

                p_lat, p_lon = point.y, point.x
                n_lat, n_lon = nearest.y, nearest.x

                if not (
                    np.isfinite(p_lat) and np.isfinite(p_lon)
                    and np.isfinite(n_lat) and np.isfinite(n_lon)
                ):
                    continue

                dist = geopy_distance((p_lat, p_lon), (n_lat, n_lon)).meters

                if dist < min_dist:
                    min_dist = dist

            except GEOSException:
                continue
            except Exception:
                continue

    return min_dist

# ----------------------------------------------------------------------
# 5. Matching function returning top candidates (unchanged)
# ----------------------------------------------------------------------
def get_top_candidates(loc,
                       stop_names, stop_lats, stop_lons,
                       stop_norms, stop_words_list, word_to_indices,
                       stop_norms_no_dir, stop_words_list_no_dir, word_to_indices_no_dir,
                       top_n=10, strict_threshold=85, fallback_threshold=70):
    """
    Return a list of up to top_n unique candidate stops (stop_name, lat, lon, score).
    Combines results from strict (with directions) and fallback (without directions) passes.
    """
    if pd.isna(loc):
        return []

    norm_loc = normalize_text(loc, expand_directions=True)
    if not norm_loc:
        return []

    loc_words = set(norm_loc.split())

    candidates_dict = {}  # idx -> max score

    # ----- Strict pass (with directions, token_sort_ratio) -----
    strict_candidates = set()
    for w in loc_words:
        strict_candidates.update(word_to_indices.get(w, set()))
    if strict_candidates:
        cand_indices = list(strict_candidates)
        cand_norms = [stop_norms[i] for i in cand_indices]
        # Get top_n matches from this set
        results = process.extract(
            norm_loc,
            cand_norms,
            scorer=fuzz.token_sort_ratio,
            limit=top_n,
            processor=None
        )
        for norm, score, _ in results:
            idx = cand_indices[cand_norms.index(norm)]
            if idx not in candidates_dict or score > candidates_dict[idx]:
                candidates_dict[idx] = score

    # ----- Fallback pass (no directions, token_set_ratio) -----
    norm_loc_no_dir = remove_direction_words(norm_loc)
    if norm_loc_no_dir:
        loc_words_no_dir = set(norm_loc_no_dir.split())
        fallback_candidates = set()
        for w in loc_words_no_dir:
            fallback_candidates.update(word_to_indices_no_dir.get(w, set()))
        if fallback_candidates:
            cand_indices_no_dir = list(fallback_candidates)
            cand_norms_no_dir = [stop_norms_no_dir[i] for i in cand_indices_no_dir]
            results = process.extract(
                norm_loc_no_dir,
                cand_norms_no_dir,
                scorer=fuzz.token_set_ratio,
                limit=top_n,
                processor=None
            )
            for norm, score, _ in results:
                idx = cand_indices_no_dir[cand_norms_no_dir.index(norm)]
                if idx not in candidates_dict or score > candidates_dict[idx]:
                    candidates_dict[idx] = score

    # Convert to list of (stop_name, lat, lon, score) sorted by score descending
    candidates_list = [
        (stop_names[idx], stop_lats[idx], stop_lons[idx], candidates_dict[idx])
        for idx in candidates_dict
    ]
    candidates_list.sort(key=lambda x: x[3], reverse=True)
    return candidates_list[:top_n]

# ----------------------------------------------------------------------
# 6. Route‑aware matching (using route_geoms dict)
# ----------------------------------------------------------------------
def match_location_with_route(loc, route_id,
                              stop_names, stop_lats, stop_lons,
                              stop_norms, stop_words_list, word_to_indices,
                              stop_norms_no_dir, stop_words_list_no_dir, word_to_indices_no_dir,
                              route_geoms,
                              top_n=10,
                              distance_threshold=200,  # meters
                              strict_threshold=85, fallback_threshold=70):
    """
    Return best stop (name, lat, lon) considering route geometry.
    Steps:
      - Get top candidates (up to top_n) from fuzzy matching.
      - For each candidate, compute distance to route geometry (if route_id exists and geometry available).
      - Select candidate with highest score among those within distance_threshold.
      - If none within threshold, take the highest‑scoring candidate and snap its coordinates
        to the nearest point on the route geometry (projection).
    """
    candidates = get_top_candidates(
        loc,
        stop_names, stop_lats, stop_lons,
        stop_norms, stop_words_list, word_to_indices,
        stop_norms_no_dir, stop_words_list_no_dir, word_to_indices_no_dir,
        top_n=top_n,
        strict_threshold=strict_threshold,
        fallback_threshold=fallback_threshold
    )
    if not candidates:
        return None, None, None

    # If route geometry is missing, just return top candidate
    if route_id not in route_geoms:
        best = candidates[0]
        return best[0], best[1], best[2]

    route_lines = route_geoms[route_id]  # list of LineStrings

    # Evaluate each candidate's distance to route
    best_in_route = None
    best_score_in_route = -1
    best_overall = candidates[0]
    best_overall_score = best_overall[3]

    for name, lat, lon, score in candidates:
        point = Point(lon, lat)
        dist = point_to_route_distance(point, route_lines)
        if dist <= distance_threshold:
            if score > best_score_in_route:
                best_in_route = (name, lat, lon)
                best_score_in_route = score

    if best_in_route is not None:
        return best_in_route
    else:
        # No candidate within threshold → snap the best overall to the route
        best_name, best_lat, best_lon, best_score = best_overall
        point = Point(best_lon, best_lat)
        # Find nearest point on route to this best candidate
        min_dist = float('inf')
        nearest_point = None
        for line in route_lines:
            if line is None:
                continue
        
            try:
                if line.is_empty:
                    continue
            except Exception:
                continue
        
            if isinstance(line, MultiLineString):
                sub_lines = list(line.geoms)
            else:
                sub_lines = [line]
        
            for sub_line in sub_lines:
                try:
                    if sub_line is None or sub_line.is_empty:
                        continue
        
                    if sub_line.length == 0:
                        continue
        
                    coords = list(sub_line.coords)
                    if len(coords) < 2:
                        continue
        
                    has_bad_coord = False
                    for coord in coords:
                        x, y = coord[:2]
                        if not np.isfinite(x) or not np.isfinite(y):
                            has_bad_coord = True
                            break
        
                    if has_bad_coord:
                        continue
        
                    nearest_on_line = nearest_points(point, sub_line)[1]
        
                    p_lat, p_lon = point.y, point.x
                    n_lat, n_lon = nearest_on_line.y, nearest_on_line.x
        
                    if not (
                        np.isfinite(p_lat) and np.isfinite(p_lon)
                        and np.isfinite(n_lat) and np.isfinite(n_lon)
                    ):
                        continue
        
                    dist = geopy_distance((p_lat, p_lon), (n_lat, n_lon)).meters
        
                    if dist < min_dist:
                        min_dist = dist
                        nearest_point = nearest_on_line
        
                except GEOSException:
                    continue
                except Exception:
                    continue
        if nearest_point is not None:
            # Use projected coordinates
            return best_name, nearest_point.y, nearest_point.x
        else:
            # Fallback (should never happen)
            return best_name, best_lat, best_lon

# ----------------------------------------------------------------------
# 7. Main function: add stop coordinates using GTFS data
# ----------------------------------------------------------------------
def add_stop_coordinates_with_route(df, location_column='Location', route_column='Route',
                                    gtfs_data=None,  # optional pre‑downloaded dict from download_gtfs_files()
                                    top_n=10, distance_threshold=200,
                                    strict_threshold=85, fallback_threshold=70):
    """
    Add stop_name, stop_lat, stop_lon columns to df, using route geometry from GTFS.
    """
    if gtfs_data is None:
        gtfs_data = download_gtfs_files()

    stops_df = gtfs_data['stops']
    trips_df = gtfs_data['trips']
    shapes_df = gtfs_data['shapes']

    # Build stop matcher
    matcher_data = build_stop_matcher(stops_df)

    # Build route geometries
    route_geoms = build_route_geometries(trips_df, shapes_df)

    def apply_match(row):
        loc = row[location_column]
        route = row[route_column]
        route_str = str(route) if pd.notna(route) else None
        name, lat, lon = match_location_with_route(
            loc, route_str, *matcher_data, route_geoms,
            top_n=top_n, distance_threshold=distance_threshold,
            strict_threshold=strict_threshold, fallback_threshold=fallback_threshold
        )
        return pd.Series([name, lat, lon])

    matched = df.apply(apply_match, axis=1)
    matched.columns = ['stop_name', 'stop_lat', 'stop_lon']
    result = pd.concat([df, matched], axis=1)
    return result

In [42]:
un = df[['Location','Route']].drop_duplicates()

un = add_stop_coordinates_with_route(
    un,
    location_column='Location',
    route_column='Route'
)

print(f"Matched rows: {un['stop_name'].notna().sum()} / {len(un)} ({un['stop_name'].notna().mean()*100:.1f}%)")

📥 Downloading GTFS ZIP from: https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/b811ead4-6eaf-4adb-8408-d389fb5a069c/resource/c920e221-7a1c-488b-8c5b-6d8cd4e85eaf/download/completegtfs.zip
✅ Extracted stops.txt
✅ Extracted trips.txt


C:\Users\bains\AppData\Local\Temp\ipykernel_75572\1860394926.py:57: DtypeWarning: Columns (0: trip_id, 1: trip_short_name) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframes[file.replace('.txt', '')] = pd.read_csv(f)


✅ Extracted shapes.txt
✅ Built geometries for 227 routes
Matched rows: 331861 / 341641 (97.1%)


In [43]:
df = df.merge(
    un[['Location', 'Route', 'stop_name', 'stop_lat', 'stop_lon']],
    on=['Location', 'Route'],
    how='left'
)

In [44]:
df_location = df[df['stop_name'].notnull()]

In [45]:
df_location

,Date,Time,Day,Location,Incident,Min Delay,Min Gap,Route,Direction,Vehicle,Transit,Year,Month,Weekday,Hour,Incident_Category,route_long_name,stop_name,stop_lat,stop_lon
0,2014-01-01,00:23:00,Wednesday,YORK MILLS STATION,Mechanical,10.0,20.0,95A,E,1734.0,Bus,2014,1,Wednesday,0.0,Mechanical,York Mills,York Mills Station at Bus Bay 1,43.745172,-79.405930
1,2014-01-01,00:23:00,Wednesday,YORK MILLS STATION,Mechanical,10.0,20.0,95B,E,1734.0,Bus,2014,1,Wednesday,0.0,Mechanical,York Mills,York Mills Station at Bus Bay 1,43.745172,-79.405930
2,2014-01-01,00:55:00,Wednesday,ENTIRE RUN FOR ROUTE,General Delay,33.0,66.0,102A,b/w,8110.0,Bus,2014,1,Wednesday,0.0,General,Markham Rd,Baycrest Home For Aged,43.729774,-79.434176
3,2014-01-01,00:55:00,Wednesday,ENTIRE RUN FOR ROUTE,General Delay,33.0,66.0,102B,b/w,8110.0,Bus,2014,1,Wednesday,0.0,General,Markham Rd,Baycrest Home For Aged,43.729774,-79.434176
4,2014-01-01,00:55:00,Wednesday,ENTIRE RUN FOR ROUTE,General Delay,33.0,66.0,102C,b/w,8110.0,Bus,2014,1,Wednesday,0.0,General,Markham Rd,Baycrest Home For Aged,43.729774,-79.434176
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1870229,2026-05-28,15:38,Thursday,JANE FWLRT STOP,EXNEA,6,12,6,E,0,LRT,2026,5,Thursday,15.0,Mechanical,Finch West Line,Jane,43.755928,-79.524287
1870230,2026-05-28,18:15,Thursday,FINCH WEST FWLRT STATI,SXUT,5,11,6,NaN,0,LRT,2026,5,Thursday,18.0,General,Finch West Line,Finch West Station - Subway Platform,43.763147,-79.490703
1870231,2026-05-29,07:41,Friday,JANE FWLRT STOP,EXNEA,7,14,6,E,0,LRT,2026,5,Friday,7.0,Mechanical,Finch West Line,Jane,43.755928,-79.524287
1870232,2026-05-29,20:09,Friday,DUNCANWOODS STOP,MXO,5,15,6,E,6515,LRT,2026,5,Friday,20.0,General,Finch West Line,Duncanwoods,43.748909,-79.557648


In [46]:
import os
import pandas as pd

# Assuming df_location is already loaded with required columns:
# stop_name, stop_lat, stop_lon, Year, Transit, Incident_Category, Min Delay

# ----------------------------------------------------------------------
# Aggregate by stop, year, transit, incident category
# ----------------------------------------------------------------------
# Ensure Year is integer (optional)
df_location['Year'] = pd.to_numeric(df_location['Year'], errors='coerce').astype('Int64')

# Group and aggregate
location_agg = df_location.groupby(
    ['stop_name', 'stop_lat', 'stop_lon', 'Year', 'Transit', 'Incident_Category'],
    as_index=False
).agg(
    Delay_Count=('Min Delay', 'count'),
    Total_Delay_Min=('Min Delay', 'sum')
)

# Optionally, remove rows where stop_name is null (if you only want matched locations)
# location_agg = location_agg.dropna(subset=['stop_name'])

# ----------------------------------------------------------------------
# Save to assets/data/location_analysis.csv
# ----------------------------------------------------------------------
output_dir = os.path.join('assets', 'data')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'location_analysis.csv')
location_agg.to_csv(output_path, index=False)

print(f"✅ location_analysis.csv saved to {output_path}")
print(f"Shape: {location_agg.shape}")
print(location_agg.head())

✅ location_analysis.csv saved to assets\data\location_analysis.csv
Shape: (107374, 8)
                         stop_name   stop_lat   stop_lon  Year    Transit  \
0  1 Front St West - Union Station  43.646205 -79.377816  2017        Bus   
1  1 Front St West - Union Station  43.646205 -79.377816  2021  Streetcar   
2  1 Front St West - Union Station  43.646205 -79.377816  2023  Streetcar   
3  1 Front St West - Union Station  43.646205 -79.377816  2026        Bus   
4  1 Front St West - Union Station  43.646205 -79.377816  2026        Bus   

  Incident_Category  Delay_Count Total_Delay_Min  
0        Operations            1            16.0  
1         Passenger            2            30.0  
2         Passenger            1            13.0  
3           General            2            25.0  
4        Operations            2            17.0  


In [ ]:
#output_path = os.path.join(output_dir, 'full_delay_data.csv')
#df.to_csv(output_path, index=False)

In [47]:
#!/usr/bin/env python3
"""
prepare_data.py

Reads location_analysis.csv and the GeoJSON boundary files from assets/data/,
performs spatial joins, and writes aggregated JSON files:
- wards_aggregated.json
- neighbourhoods_aggregated.json
- hotspots_aggregated.json

These files are optimised for client‑side filtering in the TTC Delay Analytics app.
"""

import os
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
DATA_DIR = "assets/data"          # relative to where the script is run
LOCATION_CSV = os.path.join(DATA_DIR, "location_analysis.csv")
WARD_GEOJSON = os.path.join(DATA_DIR, "gtawards.geojson")
NEIGHBOURHOOD_GEOJSON = os.path.join(DATA_DIR, "toronto.geojson")

OUT_WARDS = os.path.join(DATA_DIR, "wards_aggregated.json")
OUT_NEIGHBOURHOODS = os.path.join(DATA_DIR, "neighbourhoods_aggregated.json")
OUT_HOTSPOTS = os.path.join(DATA_DIR, "hotspots_aggregated.json")

# ----------------------------------------------------------------------
# Load location data
# ----------------------------------------------------------------------
print("📥 Loading location data...")
df = pd.read_csv(LOCATION_CSV)

# Ensure numeric columns are proper numbers
df['stop_lat'] = pd.to_numeric(df['stop_lat'], errors='coerce')
df['stop_lon'] = pd.to_numeric(df['stop_lon'], errors='coerce')
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')   # allows NaN if missing
df['Delay_Count'] = pd.to_numeric(df['Delay_Count'], errors='coerce').fillna(0).astype(int)
df['Total_Delay_Min'] = pd.to_numeric(df['Total_Delay_Min'], errors='coerce').fillna(0)

# Drop rows with invalid coordinates
df = df.dropna(subset=['stop_lat', 'stop_lon'])
print(f"   Loaded {len(df)} records with valid coordinates.")

# Create geometry column for spatial joins
geometry = [Point(xy) for xy in zip(df['stop_lon'], df['stop_lat'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# ----------------------------------------------------------------------
# Wards aggregation
# ----------------------------------------------------------------------
print("\n🏛️ Processing wards...")
wards_gdf = gpd.read_file(WARD_GEOJSON)

# Ensure both are in the same CRS (they are, but just in case)
if wards_gdf.crs != gdf.crs:
    wards_gdf = wards_gdf.to_crs(gdf.crs)

# Spatial join: assign ward name to each point (inner join – keep only points inside a ward)
joined_wards = gpd.sjoin(gdf, wards_gdf[['AREA_NAME', 'geometry']], how='inner', predicate='within')

# Group by ward, year, transit, incident category
wards_agg = joined_wards.groupby(['AREA_NAME', 'Year', 'Transit', 'Incident_Category']).agg({
    'Delay_Count': 'sum',
    'Total_Delay_Min': 'sum'
}).reset_index()

# Rename columns to match frontend expectations
wards_agg.rename(columns={
    'AREA_NAME': 'ward',
    'Year': 'year',
    'Transit': 'transit',
    'Incident_Category': 'category',
    'Delay_Count': 'delay_count',
    'Total_Delay_Min': 'total_delay_min'
}, inplace=True)

# Replace NaN years (if any) with null – they will be filtered out later anyway
wards_agg['year'] = wards_agg['year'].where(pd.notna(wards_agg['year']), None)

print(f"   Created {len(wards_agg)} aggregated ward records.")

# Write to JSON
wards_agg.to_json(OUT_WARDS, orient='records', indent=None)   # compact JSON
print(f"   ✅ Saved to {OUT_WARDS}")

# ----------------------------------------------------------------------
# Neighbourhoods aggregation
# ----------------------------------------------------------------------
print("\n🏘️ Processing neighbourhoods...")
neighbourhoods_gdf = gpd.read_file(NEIGHBOURHOOD_GEOJSON)

if neighbourhoods_gdf.crs != gdf.crs:
    neighbourhoods_gdf = neighbourhoods_gdf.to_crs(gdf.crs)

joined_neigh = gpd.sjoin(gdf, neighbourhoods_gdf[['AREA_NAME', 'geometry']], how='inner', predicate='within')

neigh_agg = joined_neigh.groupby(['AREA_NAME', 'Year', 'Transit', 'Incident_Category']).agg({
    'Delay_Count': 'sum',
    'Total_Delay_Min': 'sum'
}).reset_index()

neigh_agg.rename(columns={
    'AREA_NAME': 'neighbourhood',
    'Year': 'year',
    'Transit': 'transit',
    'Incident_Category': 'category',
    'Delay_Count': 'delay_count',
    'Total_Delay_Min': 'total_delay_min'
}, inplace=True)

neigh_agg['year'] = neigh_agg['year'].where(pd.notna(neigh_agg['year']), None)

print(f"   Created {len(neigh_agg)} aggregated neighbourhood records.")
neigh_agg.to_json(OUT_NEIGHBOURHOODS, orient='records', indent=None)
print(f"   ✅ Saved to {OUT_NEIGHBOURHOODS}")

# ----------------------------------------------------------------------
# Hotspots aggregation (by unique stop)
# ----------------------------------------------------------------------
print("\n🔥 Processing hotspots (stops)...")
# Group by stop coordinates, year, transit, category
hotspot_agg = df.groupby(['stop_lat', 'stop_lon', 'Year', 'Transit', 'Incident_Category']).agg({
    'Delay_Count': 'sum',
    'Total_Delay_Min': 'sum'
}).reset_index()

hotspot_agg.rename(columns={
    'stop_lat': 'lat',
    'stop_lon': 'lon',
    'Year': 'year',
    'Transit': 'transit',
    'Incident_Category': 'category',
    'Delay_Count': 'delay_count',
    'Total_Delay_Min': 'total_delay_min'
}, inplace=True)

hotspot_agg['year'] = hotspot_agg['year'].where(pd.notna(hotspot_agg['year']), None)

print(f"   Created {len(hotspot_agg)} aggregated hotspot records.")
hotspot_agg.to_json(OUT_HOTSPOTS, orient='records', indent=None)
print(f"   ✅ Saved to {OUT_HOTSPOTS}")

print("\n🎉 All aggregations complete!")

📥 Loading location data...
   Loaded 107374 records with valid coordinates.

🏛️ Processing wards...
   Created 5264 aggregated ward records.
   ✅ Saved to assets/data\wards_aggregated.json

🏘️ Processing neighbourhoods...
   Created 17016 aggregated neighbourhood records.
   ✅ Saved to assets/data\neighbourhoods_aggregated.json

🔥 Processing hotspots (stops)...
   Created 102965 aggregated hotspot records.
   ✅ Saved to assets/data\hotspots_aggregated.json

🎉 All aggregations complete!


In [48]:
import requests
import pandas as pd
import zipfile
import json
import os
from io import BytesIO, StringIO

def generate_route_geometries():
    """
    Downloads trips.txt and shapes.txt from the merged GTFS package,
    and generates a JSON file mapping route+short_name combinations
    to their shape geometries (list of [lat, lon] points).
    The file is saved as assets/data/route_geometries.json.
    """
    base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
    package_id = "merged-gtfs-ttc-routes-and-schedules"
    output_dir = os.path.join("assets", "data")
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, "route_geometries.json")

    print("\n🗺️ Generating route geometries from GTFS...")

    # 1. Get package metadata
    package_url = f"{base_url}/api/3/action/package_show"
    resp = requests.get(package_url, params={"id": package_id})
    resp.raise_for_status()
    data = resp.json()
    if not data.get("success"):
        raise Exception("CKAN API request failed for GTFS package")

    # 2. Find the ZIP resource (non-datastore, format=ZIP)
    zip_resource = None
    for res in data["result"]["resources"]:
        name = res.get("name", "").lower()
        fmt = res.get("format", "").lower()
        if "gtfs" in name and fmt == "zip":
            zip_resource = res
            break

    if not zip_resource:
        raise Exception("No GTFS ZIP resource found in package")

    print(f"📥 Downloading GTFS ZIP from: {zip_resource['url']}")
    zip_resp = requests.get(zip_resource['url'])
    zip_resp.raise_for_status()

    # 3. Extract trips.txt and shapes.txt into memory
    required_files = ['trips.txt', 'shapes.txt']
    gtfs_data = {}
    with zipfile.ZipFile(BytesIO(zip_resp.content)) as zf:
        for fname in required_files:
            if fname in zf.namelist():
                with zf.open(fname) as f:
                    gtfs_data[fname] = f.read().decode('utf-8')
                print(f"✅ Extracted {fname}")
            else:
                print(f"⚠️ {fname} not found in ZIP")
                return

    # 4. Read trips.txt
    trips = pd.read_csv(StringIO(gtfs_data['trips.txt']))
    # Keep only necessary columns: route_id, trip_short_name, shape_id
    trips = trips[['route_id', 'trip_short_name', 'shape_id']].drop_duplicates()
    # Convert to string and fill missing short names
    trips['route_id'] = trips['route_id'].astype(str)
    trips['trip_short_name'] = trips['trip_short_name'].fillna('').astype(str).str.strip()

    # 5. Read shapes.txt
    shapes = pd.read_csv(StringIO(gtfs_data['shapes.txt']))
    shapes = shapes[['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence']]
    shapes = shapes.sort_values(['shape_id', 'shape_pt_sequence'])

    # 6. Build geometry per shape_id
    shape_geometries = {}
    for shape_id, group in shapes.groupby('shape_id'):
        # Create list of [lat, lon] pairs in order
        coords = group[['shape_pt_lat', 'shape_pt_lon']].values.tolist()
        shape_geometries[shape_id] = coords

    # 7. Merge trips with shape geometries
    # For each (route_id, trip_short_name) we need a geometry.
    # There might be multiple trips with same route+short_name but different shape_id.
    # We'll take the first shape_id for each combination (assuming consistency).
    route_geometries = {}
    # Group trips by (route_id, trip_short_name) and take first shape_id
    for (route_id, short_name), group in trips.groupby(['route_id', 'trip_short_name']):
        shape_id = group.iloc[0]['shape_id']  # first shape_id
        if shape_id in shape_geometries:
            key = route_id + short_name  # e.g., "100A"
            route_geometries[key] = shape_geometries[shape_id]
        else:
            print(f"⚠️ Shape ID {shape_id} not found for route {key}")

    # 8. Save to JSON
    with open(output_file, 'w') as f:
        json.dump(route_geometries, f, indent=2)
    print(f"✅ Route geometries saved to {output_file}")
    return route_geometries

# ----------------------------------------------------------------------
# Example usage
# ----------------------------------------------------------------------
if __name__ == "__main__":
    generate_route_geometries()


🗺️ Generating route geometries from GTFS...
📥 Downloading GTFS ZIP from: https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/b811ead4-6eaf-4adb-8408-d389fb5a069c/resource/c920e221-7a1c-488b-8c5b-6d8cd4e85eaf/download/completegtfs.zip
✅ Extracted trips.txt
✅ Extracted shapes.txt


C:\Users\bains\AppData\Local\Temp\ipykernel_75572\1775921265.py:61: DtypeWarning: Columns (0: trip_id, 1: trip_short_name) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv(StringIO(gtfs_data['trips.txt']))


⚠️ Shape ID nan not found for route 49S
✅ Route geometries saved to assets\data\route_geometries.json
